In [1]:
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import date, datetime, timedelta

import os

import duckdb
con = duckdb.connect("pbp.duckdb")
# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## Historical Data Up to Season End

### Game Summary

In [2]:
history_summary = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Base Data\Game Summary\all_game_summary.csv")



In [ ]:

history_summary = history_summary[history_summary['season_id']!=20252026]
history_summary = history_summary[['game_id', 'season_id', 'awayTeam.id', 'awayTeam.abbrev', 'homeTeam.id', 'homeTeam.abbrev', 'awayTeam.score',
                                   'homeTeam.score', 'gameState', 'link', 'game_date_time', 'game_date']]
history_summary.head()

,game_id,season_id,awayTeam.id,awayTeam.abbrev,homeTeam.id,homeTeam.abbrev,awayTeam.score,homeTeam.score,gameState,link,game_date_time,game_date
0,2013020001,20132014,10,TOR,8,MTL,4,3,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 19:00:00-04:00,10/1/2013
1,2013020002,20132014,15,WSH,16,CHI,4,6,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 20:00:00-04:00,10/1/2013
2,2013020003,20132014,52,WPG,22,EDM,5,4,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 22:00:00-04:00,10/1/2013
3,2013020004,20132014,10,TOR,4,PHI,3,1,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-02 19:30:00-04:00,10/2/2013
4,2013020005,20132014,7,BUF,17,DET,1,2,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-02 20:00:00-04:00,10/2/2013


In [4]:
current_summary = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Thanksgiving Prep\current_game_summary.csv")
current_summary = current_summary.rename(columns={'season':'season_id'})
current_summary = current_summary[['game_id', 'season_id', 'awayTeam.id', 'awayTeam.abbrev', 'homeTeam.id', 'homeTeam.abbrev', 'awayTeam.score',
                                   'homeTeam.score', 'gameState', 'link', 'game_date_time', 'game_date']]
current_summary.tail()

,game_id,season_id,awayTeam.id,awayTeam.abbrev,homeTeam.id,homeTeam.abbrev,awayTeam.score,homeTeam.score,gameState,link,game_date_time,game_date
2754,2025020431,20252026,10,TOR,12,CAR,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-12-04 19:00:00-05:00,2025-12-04
2755,2025020432,20252026,17,DET,29,CBJ,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-12-04 19:30:00-05:00,2025-12-04
2756,2025020433,20252026,30,MIN,20,CGY,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-12-04 21:00:00-05:00,2025-12-04
2757,2025020434,20252026,55,SEA,22,EDM,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-12-04 21:00:00-05:00,2025-12-04
2758,2025020435,20252026,16,CHI,26,LAK,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-12-04 22:00:00-05:00,2025-12-04


### Standings Data

In [5]:



# added_standings = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\1920__standings.csv")
# added_standings = added_standings[['seasonId','tri_code', 'team_name', 'conferenceName', 'conferenceSequence', 
#        'gamesPlayed', 'points', 'winPctg', 'pointPctg', 'regulationPlusOtWins',
#        'goalDifferential', 'goalFor', 'goalAgainst', 'homeGamesPlayed',
#        'homeWins', 'roadGamesPlayed', 'roadWins', 'l10GoalDifferential',
#        'streakCode', 'streakCount', 'home_win_pctg', 'road_win_pctg', 'gf_g', 'ga_g', 'projected_points']]
# added_standings = added_standings.rename(columns={'seasonId':'season_id'})
# added_standings.head()

In [6]:

final_standings = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\history_final_standings.csv")
# final_standings = pd.concat([final_standings, added_standings])


final_standings.head()

,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points
0,20132014,BOS,Boston Bruins,Eastern,1,82,117,0.658537,0.713415,51,84,261,177,41,31,41,23,7,L,1,0.756098,0.560976,3.182927,2.158537,117.0
1,20132014,ANA,Anaheim Ducks,Western,1,82,116,0.658537,0.707317,51,57,266,209,41,29,41,25,8,W,4,0.707317,0.609756,3.243902,2.548780,116.0
2,20132014,COL,Colorado Avalanche,Western,2,82,112,0.634146,0.682927,47,30,250,220,41,26,41,26,7,OT,1,0.634146,0.634146,3.048780,2.682927,112.0
3,20132014,STL,St. Louis Blues,Western,3,82,111,0.634146,0.676829,43,57,248,191,41,28,41,24,-13,L,6,0.682927,0.585366,3.024390,2.329268,111.0
4,20132014,SJS,San Jose Sharks,Western,4,82,111,0.621951,0.676829,41,49,249,200,41,29,41,22,1,W,2,0.707317,0.536585,3.036585,2.439024,111.0


In [7]:
# tx_added = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\1920_tx_standings.csv")
# tx_added = tx_added[['seasonId','tri_code', 'team_name', 'conferenceName', 'conferenceSequence', 
#        'gamesPlayed', 'points', 'winPctg', 'pointPctg', 'regulationPlusOtWins',
#        'goalDifferential', 'goalFor', 'goalAgainst', 'homeGamesPlayed',
#        'homeWins', 'roadGamesPlayed', 'roadWins', 'l10GoalDifferential',
#        'streakCode', 'streakCount', 'home_win_pctg', 'road_win_pctg', 'gf_g', 'ga_g', 'projected_points']]
# tx_added = tx_added.rename(columns={'seasonId':'season_id'})


In [8]:
thanksgiving_standings_df = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\thanksgiving_standings_df.csv")
thanksgiving_standings_df = thanksgiving_standings_df[['seasonId','tri_code', 'team_name', 'conferenceName', 'conferenceSequence', 
       'gamesPlayed', 'points', 'winPctg', 'pointPctg', 'regulationPlusOtWins',
       'goalDifferential', 'goalFor', 'goalAgainst', 'homeGamesPlayed',
       'homeWins', 'roadGamesPlayed', 'roadWins', 'l10GoalDifferential',
       'streakCode', 'streakCount', 'home_win_pctg', 'road_win_pctg', 'gf_g', 'ga_g', 'projected_points', 'thanksgiving_date']]
thanksgiving_standings_df = thanksgiving_standings_df.rename(columns={'seasonId':'season_id'})

# thanksgiving_standings_df=pd.concat([thanksgiving_standings_df,tx_added])
thanksgiving_standings_df.head()

,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,thanksgiving_date
0,20132014,CHI,Chicago Blackhawks,Western,1,26,40,0.692308,0.769231,15,22,95,73,13,9,13,9,9,W,4,0.692308,0.692308,3.653846,2.807692,126.0,2013-11-28
1,20132014,STL,St. Louis Blues,Western,2,24,39,0.750000,0.812500,15,35,86,51,14,11,10,7,18,W,5,0.785714,0.700000,3.583333,2.125000,133.0,2013-11-28
2,20132014,SJS,San Jose Sharks,Western,3,24,37,0.666667,0.770833,14,28,82,54,11,8,13,8,2,W,3,0.727273,0.615385,3.416667,2.250000,126.0,2013-11-28
3,20132014,ANA,Anaheim Ducks,Western,4,27,37,0.629630,0.685185,16,12,83,71,10,9,17,8,-3,L,1,0.900000,0.470588,3.074074,2.629630,112.0,2013-11-28
4,20132014,LAK,Los Angeles Kings,Western,5,26,36,0.615385,0.692308,12,13,69,56,13,8,13,8,8,OT,1,0.615385,0.615385,2.653846,2.153846,114.0,2013-11-28


In [9]:
thanksgiving_standings_df['season_id'].value_counts()

season_id
20222023    32
20232024    32
20242025    32
20212022    32
20182019    31
20172018    31
20192020    31
20152016    30
20142015    30
20132014    30
20162017    30
Name: count, dtype: int64

In [10]:
current_tx_standings = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\current_thanksgiving_standings.csv")
current_tx_standings = current_tx_standings[['seasonId','tri_code', 'team_name', 'conferenceName', 'conferenceSequence', 
       'gamesPlayed', 'points', 'winPctg', 'pointPctg', 'regulationPlusOtWins',
       'goalDifferential', 'goalFor', 'goalAgainst', 'homeGamesPlayed',
       'homeWins', 'roadGamesPlayed', 'roadWins', 'l10GoalDifferential',
       'streakCode', 'streakCount', 'home_win_pctg', 'road_win_pctg', 'gf_g', 'ga_g', 'projected_points']]

current_tx_standings['thanksgiving_date'] = '2025-11-27'
current_tx_standings = current_tx_standings.rename(columns={'seasonId':'season_id'})
current_tx_standings.head()

,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,thanksgiving_date
0,20252026,COL,Colorado Avalanche,Western,1,23,39,0.739130,0.847826,17,44,94,50,11,9,12,8,32,W,10,0.818182,0.666667,4.086957,2.173913,139.0,2025-11-27
1,20252026,DAL,Dallas Stars,Western,2,24,34,0.625000,0.708333,13,16,83,67,11,6,13,9,20,W,2,0.545455,0.692308,3.458333,2.791667,116.0,2025-11-27
2,20252026,NJD,New Jersey Devils,Eastern,1,23,31,0.652174,0.673913,13,3,72,69,10,9,13,6,-3,W,2,0.900000,0.461538,3.130435,3.000000,111.0,2025-11-27
3,20252026,TBL,Tampa Bay Lightning,Eastern,2,23,30,0.608696,0.652174,14,13,74,61,13,8,10,6,12,W,5,0.615385,0.600000,3.217391,2.652174,107.0,2025-11-27
4,20252026,CAR,Carolina Hurricanes,Eastern,3,23,30,0.608696,0.652174,13,11,80,69,10,6,13,8,-2,L,2,0.600000,0.615385,3.478261,3.000000,107.0,2025-11-27


### Play by Play Data

In [11]:
# all_pbp = con.execute("Select * from game_plays").df()
# all_pbp.head()
all_pbp = con.execute("""
                    SELECT
                    season_id, 
                    game_id, 
                    eventId, 
                    timeInPeriod, 
                    timeRemaining,
                    typeCode, 
                    typeDescKey,
                    sortOrder, 
                    period_number, 
                    event_team_id as team_id,
                    x_coord,
                    y_coord, 
                    zone_code, 
                    hitter, 
                    hittee, 
                    shot_type,
                    goalie, 
                    away_sog, 
                    home_sog, 
                    player_id,
                    reason,  
                    blocking_player, 
                    scoring_player,
                    situation, 
                    goalie_situation, 
                    game_in_seconds, 
                    FROM game_plays""").df()
all_pbp['season_id'].value_counts()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

season_id
20242025    420073
20222023    407529
20232024    407529
20172018    405453
20212022    404400
20182019    401362
20142015    391839
20132014    384812
20162017    382015
20152016    381653
20192020    336025
Name: count, dtype: int64

In [12]:
# con.execute(r"""
#     CREATE TABLE current_plays AS
#     SELECT * FROM read_csv_auto(
#         'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Base Data\\Play By Play\\play_by_play_2526.csv'
#     );
# """)



current_pbp = con.execute("""
                    SELECT
                    season_id, 
                    game_id, 
                    eventId, 
                    timeInPeriod, 
                    timeRemaining,
                    typeCode, 
                    typeDescKey,
                    sortOrder, 
                    period_number, 
                    event_team_id as team_id,
                    x_coord,
                    y_coord, 
                    zone_code, 
                    hitter, 
                    hittee, 
                    shot_type,
                    goalie, 
                    away_sog, 
                    home_sog, 
                    player_id,
                    reason,  
                    blocking_player, 
                    scoring_player,
                    situation, 
                    goalie_situation, 
                    game_in_seconds, 
                    FROM current_plays""").df()
current_pbp.head()




,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds
0,20252026,2025020001,52,00:00:00,20:00:00,520,period-start,8,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,0
1,20252026,2025020001,51,00:00:00,20:00:00,502,faceoff,11,1,16.0,0.0,0.0,N,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,0
2,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22
3,20252026,2025020001,8,00:34:00,19:26:00,516,stoppage,13,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,puck-in-crowd,NaN,NaN,5 on 5,in net,34
4,20252026,2025020001,55,00:34:00,19:26:00,502,faceoff,16,1,16.0,-69.0,22.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,34


In [13]:
dfs = [history_summary, all_pbp, current_pbp]
exclude_cols = ["x_coord", "y_coord"]

for i, df in enumerate(dfs):
    float_cols = df.select_dtypes(include='float').columns
    float_cols = float_cols.difference(exclude_cols)
    dfs[i][float_cols] = df[float_cols].astype("Int64")





In [14]:
# 1. Build pandas DataFrame
home_away = history_summary[['season_id','game_id', 'homeTeam.id', 'awayTeam.id']]
home_away = home_away.rename(columns={
    'homeTeam.id': 'home_team', 
    'awayTeam.id': 'away_team'
})

# 2. Store permanently in DuckDB
con.execute("DROP TABLE IF EXISTS home_away;")
con.execute("CREATE TABLE home_away AS SELECT * FROM home_away;")

#3. Join with game_plays (the correct DuckDB table name is home_away)
game_plays = con.execute("""
    SELECT gp.season_id, 
                    gp.game_id, 
                    gp.eventId, 
                    gp.timeInPeriod, 
                    gp.timeRemaining,
                    gp.typeCode, 
                    gp.typeDescKey,
                    gp.sortOrder, 
                    gp.period_number, 
                    gp.event_team_id as team_id,
                    gp.x_coord,
                    gp.y_coord, 
                    gp.zone_code, 
                    gp.hitter, 
                    gp.hittee, 
                    gp.shot_type,
                    gp.goalie, 
                    gp.away_sog, 
                    gp.home_sog, 
                    gp.player_id,
                    gp.reason,  
                    gp.blocking_player, 
                    gp.scoring_player,
                    gp.situation, 
                    gp.goalie_situation, 
                    gp.game_in_seconds, 
                    ha.home_team,
                    ha.away_team,
                    ha.season_id,
                    ha.game_id
    FROM game_plays gp
    LEFT JOIN home_away ha
        ON gp.game_id = ha.game_id
       AND gp.season_id = ha.season_id
""").df()

game_plays.tail()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
4742758,20242025,2024021038,526,19:37:00,00:23:00,525,takeaway,289,1,24.0,-93.0,-7.0,D,NaN,NaN,None,NaN,NaN,NaN,8484153.0,None,NaN,NaN,5 on 5,in net,1177,59,24,20242025,2024021038
4742759,20242025,2024021038,519,19:42:00,00:18:00,508,blocked-shot,290,1,24.0,-64.0,-12.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8483431.0,NaN,5 on 5,in net,1182,59,24,20242025,2024021038
4742760,20242025,2024021038,202,19:43:00,00:17:00,507,missed-shot,291,1,24.0,-92.0,-18.0,O,NaN,NaN,wrist,8478872.0,NaN,NaN,NaN,failed-bank-attempt,NaN,NaN,5 on 5,in net,1183,59,24,20242025,2024021038
4742761,20242025,2024021038,26,19:44:00,00:16:00,516,stoppage,292,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,goalie-stopped-after-sog,NaN,NaN,5 on 5,in net,1184,59,24,20242025,2024021038
4742762,20242025,2024021038,73,19:44:00,00:16:00,502,faceoff,294,1,59.0,-69.0,-22.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,1184,59,24,20242025,2024021038


In [15]:
curr_home_away= current_summary[['season_id','game_id', 'homeTeam.id', 'awayTeam.id']]
curr_home_away = curr_home_away.rename(columns={'homeTeam.id':'home_team', 'awayTeam.id':'away_team'})
con.execute("DROP TABLE IF EXISTS home_away;")
con.execute("CREATE TABLE home_away AS SELECT * FROM curr_home_away;")



current_game_plays = con.execute("""
    SELECT cp.season_id, 
                    cp.game_id, 
                    cp.eventId, 
                    cp.timeInPeriod, 
                    cp.timeRemaining,
                    cp.typeCode, 
                    cp.typeDescKey,
                    cp.sortOrder, 
                    cp.period_number, 
                    cp.event_team_id as team_id,
                    cp.x_coord,
                    cp.y_coord, 
                    cp.zone_code, 
                    cp.hitter, 
                    cp.hittee, 
                    cp.shot_type,
                    cp.goalie, 
                    cp.away_sog, 
                    cp.home_sog, 
                    cp.player_id,
                    cp.reason,  
                    cp.blocking_player, 
                    cp.scoring_player,
                    cp.situation, 
                    cp.goalie_situation, 
                    cp.game_in_seconds, 
                    ha.home_team,
                    ha.away_team,
                    ha.season_id,
                    ha.game_id
    FROM current_plays cp
    LEFT JOIN home_away ha
        ON cp.game_id = ha.game_id
       AND cp.season_id = ha.season_id
""").df()
current_game_plays.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
0,20252026,2025020001,52,00:00:00,20:00:00,520,period-start,8,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,0,13,16,20252026,2025020001
1,20252026,2025020001,51,00:00:00,20:00:00,502,faceoff,11,1,16.0,0.0,0.0,N,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,0,13,16,20252026,2025020001
2,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001
3,20252026,2025020001,8,00:34:00,19:26:00,516,stoppage,13,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,puck-in-crowd,NaN,NaN,5 on 5,in net,34,13,16,20252026,2025020001
4,20252026,2025020001,55,00:34:00,19:26:00,502,faceoff,16,1,16.0,-69.0,22.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,34,13,16,20252026,2025020001


### Goalies Data

In [16]:
con.execute(r"""
    CREATE OR REPLACE TABLE all_goalie_games AS
    SELECT
        season       AS season_id,
        gameId      AS game_id,
        *
    EXCLUDE (season, gameId)
    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Goalies\\all_goalie_games.csv'
    )
""")
goalies_individual_games = con.execute("""
    SELECT *
    FROM all_goalie_games
""").df()
goalies_individual_games.tail()

,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
32172,20252026,2025020117,1,1,Joel Hofer,8480981,STL,477,0,0,0,22.64150,3,0,0.50000,6,3
32173,20252026,2025020275,1,1,Alex Lyon,8479312,BUF,370,0,0,0,19.45945,1,0,0.33333,3,2
32174,20252026,2025020341,1,1,Logan Thompson,8480313,WSH,1099,0,1,0,13.10282,2,0,0.33333,6,4
32175,20252026,2025020362,1,1,Dustin Wolf,8481692,CGY,352,0,1,0,30.68181,1,0,0.25000,4,3
32176,20252026,2025020259,1,0,Pyotr Kochetkov,8481611,CAR,216,0,0,0,0.00000,0,0,NaN,0,0


### Special Teams

In [17]:
# all_pk = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pk_by_game_all.csv")
# all_pp = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pp_by_game_all.csv")

# all_pp = all_pp.rename(columns={'season':'season_id', 'gameId':'game_id'}).sort_values(by='game_id')
# all_pk = all_pk.rename(columns={'season':'season_id', 'gameId':'game_id'}).sort_values(by='game_id')

In [18]:

con.execute(r"""
    CREATE OR REPLACE TABLE all_pp AS
    SELECT
        season       AS season_id,
        gameId       AS game_id,
        teamId       AS team_id,
        *
    EXCLUDE (season, gameId, teamId)
    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Team Stats\\pp_by_game_all.csv'
    )
    ORDER BY game_id;
""")



con.execute(r"""
    CREATE OR REPLACE TABLE all_pk AS
    SELECT
        season       AS season_id,
        gameId       AS game_id,
        teamId       AS team_id,
        *
    EXCLUDE (season, gameId,teamId)
    FROM read_csv_auto(
        'C:\\Users\\kjchr\\Desktop\\hockey\\new_hockey_env\\API data\\Team Stats\\pk_by_game_all.csv'
    )
    ORDER BY game_id;
""")

all_pp = con.execute("""
    SELECT *
    FROM all_pp
""").df()

all_pk = con.execute("""
    SELECT *
    FROM all_pk
""").df()


### pbp clean

In [19]:
# # # ### concatenate the seasons together###
# play_by_play_1314 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1314.csv")
# play_by_play_1415 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1415.csv")
# play_by_play_1516 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1516.csv")
# play_by_play_1617 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1617.csv")
# play_by_play_1718 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1718.csv")
# play_by_play_1819 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1819.csv")
# play_by_play_1920 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_1920.csv")
# play_by_play_2021 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_2021.csv")
# play_by_play_2122 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_2122.csv")
# play_by_play_2223 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_2223.csv")
# play_by_play_2324 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_2324.csv")
# play_by_play_2425 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_2425.csv")


# # play_by_play_1314['season_id']= 20132014
# # play_by_play_1415['season_id']= 20142015
# # play_by_play_1516['season_id']= 20152016
# # play_by_play_1617['season_id']= 20162017
# # play_by_play_1718['season_id']= 20172018
# # play_by_play_1819['season_id']= 20182019
# # play_by_play_1920['season_id']= 20192020
# # play_by_play_2021['season_id']= 20202021
# # play_by_play_2122['season_id']= 20212022
# # play_by_play_2223['season_id']= 20222023
# # play_by_play_2324['season_id']= 20232024
# play_by_play_2425['season_id']= 20242025


# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\play_by_play_2425.csv"

# # Ensure the folder exists before writing the file
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # # Export the data to CSV
# play_by_play_2425.to_csv(export_path, index=False)


# # expected_cols = ['season_id', 'game_id', 'eventId', 'timeInPeriod', 'timeRemaining', 'situationCode',
# #        'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder',
# #        'period_number', 'period_type', 'max_regulation_periods',
# #        'event_team_id', 'details.losingPlayerId', 'winning_player_id',
# #        'x_coord', 'y_coord', 'zone_code', 'hitter', 'hittee', 'shot_type',
# #        'shooting_player', 'goalie', 'away_sog', 'home_sog', 'player_id',
# #        'reason', 'blocking_player', 'type_code', 'desc_key', 'duration',
# #        'committed_by', 'drawn_by', 'secondary_reason', 'pptReplayUrl',
# #        'scoring_player', 'scoring_player_total', 'assist_1', 'assist1_total',
# #        'away_score', 'home_score', 'details.highlightClipSharingUrl',
# #        'details.highlightClip', 'details.discreteClip', 'assist_2',
# #        'assist2_total', 'served_by', 'details.discreteClipFr',
# #        'details.highlightClipSharingUrlFr', 'details.highlightClipFr',
# #        'situation', 'goalie_situation', 'game_in_seconds']
# # dfs = [play_by_play_1314, play_by_play_1415, play_by_play_1516, play_by_play_1617, play_by_play_1718, play_by_play_1819, play_by_play_1920,
# #        play_by_play_2021, play_by_play_2122,play_by_play_2223,play_by_play_2324, play_by_play_2425]
# # for name, df in zip(
# #     ["1314","1415","1516","1617","1718","1819","1920","2122","2223","2324","2425"],
# #     dfs
# # ):
# #     df_cols = df.columns.tolist()

# #     missing = [c for c in expected_cols if c not in df_cols]
# #     extra   = [c for c in df_cols if c not in expected_cols]

# #     print(f"\n===== Season {name} =====")
# #     print("Missing columns:", missing if missing else "None")
# #     print("Extra columns:",   extra   if extra   else "None")
# # # for name, df in zip(
# # #     ["1314","1415","1516","1617","1718","1819","1920","2021","2122","2223","2324","2425"],
# # #     dfs
# # # ):
# # #     print(f"\n===== {name} COLUMNS =====")
# # #     for col in df.columns:
# # #         print(repr(col))
# # # print(len(dfs))
# # # print([type(x) for x in dfs])


# # # aligned_dfs = [df.reindex(columns=expected_cols) for df in dfs]
# # # play_by_play = pd.concat(aligned_dfs, ignore_index=True)


In [20]:
# ### concatenate the seasons together###
# gs_1314 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1314.csv")
# gs_1415 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1415.csv")
# gs_1516 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1516.csv")
# gs_1617 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1617.csv")
# gs_1718 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1718.csv")
# gs_1819 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1819.csv")
# gs_1920 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_1920.csv")
# gs_2021 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_2021.csv")
# gs_2122 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_2122.csv")
# gs_2223 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_2223.csv")
# gs_2324 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_2324.csv")
# gs_2425 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\summary_2425.csv")
# gs_2526 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Game Summary\current_game_summary.csv")

# expected_cols = ['game_id', 'season_id', 'gameType', 'awayTeam.id', 'awayTeam.abbrev',
#        'awayTeam.logo', 'homeTeam.id', 'homeTeam.abbrev', 'homeTeam.logo',
#        'homeTeam.placeName.default', 'awayTeam.placeName.default',
#        'awayTeam.score', 'homeTeam.score', 'winningGoalScorer.playerId',
#        'winningGoalie.playerId', 'gameState', 'link', 'game_date_time',
#        'start_time', 'game_date']


# dfs = [gs_1314, gs_1415, gs_1516, gs_1617,gs_1718,gs_1819,gs_1920,gs_2021,gs_2122,gs_2223,gs_2425,gs_2526]

# aligned_dfs = [df.reindex(columns=expected_cols) for df in dfs]
# all_game_summary = pd.concat(aligned_dfs, ignore_index=True)


In [21]:

# home_away= current_game_summary[['season','game_id', 'homeTeam.id', 'awayTeam.id']]
# # home_away = home_away.rename(columns={'homeTeam.id':'home_team', 'awayTeam.id':'away_team'})

In [22]:

# # export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Base Data\Game Summary\all_game_summary"

# # # Ensure the folder exists before writing the file
# # os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # all_game_summary.to_csv(export_path, index=False)

# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Play By Play\history_play_by_play.csv"

# # Ensure the folder exists before writing the file
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# play_by_play.to_csv(export_path, index=False)

## Historical Data Up to Thanksgiving

In [23]:
#THANKSGIVING CUTOFFS
calendar = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Season Dates Info\season_calendar_dates.csv")

thanksgiving = calendar[['season_id', 'thanksgiving_date']]
# # thanksgiving_games = thanksgiving.merge(history_summary, on='season_id', how='inner')
thanksgiving.head()

,season_id,thanksgiving_date
0,20002001,2000-11-23
1,20012002,2001-11-22
2,20022003,2002-11-28
3,20032004,2003-11-24
4,20052006,2005-11-24


In [25]:

# thanksgiving.head()
merged = history_summary.merge(
    thanksgiving[['season_id', 'thanksgiving_date']],
    on='season_id',
    how='left'
)
thanksgiving_game_summary = merged[ merged['game_date'] <= merged['thanksgiving_date'] ].copy()
thanksgiving_game_summary.head()

,game_id,season_id,awayTeam.id,awayTeam.abbrev,homeTeam.id,homeTeam.abbrev,awayTeam.score,homeTeam.score,gameState,link,game_date_time,game_date,thanksgiving_date
0,2013020001,20132014,10,TOR,8,MTL,4,3,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 19:00:00-04:00,10/1/2013,2013-11-28
1,2013020002,20132014,15,WSH,16,CHI,4,6,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 20:00:00-04:00,10/1/2013,2013-11-28
2,2013020003,20132014,52,WPG,22,EDM,5,4,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-01 22:00:00-04:00,10/1/2013,2013-11-28
3,2013020004,20132014,10,TOR,4,PHI,3,1,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-02 19:30:00-04:00,10/2/2013,2013-11-28
4,2013020005,20132014,7,BUF,17,DET,1,2,OFF,https://api-web.nhle.com/v1/gamecenter/2013020...,2013-10-02 20:00:00-04:00,10/2/2013,2013-11-28


In [26]:
tx_teams = thanksgiving_game_summary[['homeTeam.id', 'homeTeam.abbrev']]
tx_teams=tx_teams.drop_duplicates()
tx_teams = tx_teams.rename(columns={'homeTeam.id':'team_id', 'homeTeam.abbrev':'tri_code'})
tx_teams.head()

,team_id,tri_code
0,8,MTL
1,16,CHI
2,22,EDM
3,4,PHI
4,17,DET


In [6]:
teams = current_summary[['homeTeam.id', 'homeTeam.abbrev']]
teams=teams.drop_duplicates()
teams = teams.rename(columns={'homeTeam.id':'team_id', 'homeTeam.abbrev':'tri_code'})
teams.head()

,team_id,tri_code
0,13,FLA
1,3,NYR
2,26,LAK
3,10,TOR
5,15,WSH


In [ ]:

# # Define export path
# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Thanksgiving Prep\teams_list.csv"

# # Ensure the folder exists before writing the file
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # # Export the data to CSV
# teams.to_csv(export_path, index=False)


In [28]:
thanksgiving_game_ids = thanksgiving_game_summary['game_id'].tolist()
game_ids = current_summary['game_id'].tolist()

### Goalies Data

In [29]:
tx_goalies_individual = goalies_individual_games[
    goalies_individual_games['game_id'].isin(thanksgiving_game_ids)
]

tx_goalies_individual.head()


,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
0,20192020,2019020796,1,1,Henrik Lundqvist,8468685,NYR,3600,1,0,0,0.0,33,1,1.0,33,0
1,20192020,2019020805,1,1,Curtis McElhinney,8470147,TBL,3583,1,0,0,0.0,30,1,1.0,30,0
2,20192020,2019020905,1,1,Marc-Andre Fleury,8470594,VGK,3600,1,0,0,0.0,19,1,1.0,19,0
3,20192020,2019020118,1,1,Marc-Andre Fleury,8470594,VGK,3575,1,0,0,0.0,29,1,1.0,29,0
4,20192020,2019020315,1,1,Marc-Andre Fleury,8470594,VGK,3598,1,0,0,0.0,34,1,1.0,34,0


In [30]:
tx_goalies_indiv = tx_goalies_individual.groupby(['goalieFullName','season_id','teamAbbrev']).agg(
    i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * tx_goalies_individual.loc[x.index, 'shotsAgainst']).replace(0, np.nan).sum() / tx_goalies_individual.loc[x.index, 'shotsAgainst'].replace(0, np.nan).sum()),
    i_GAA=pd.NamedAgg(column='goalsAgainstAverage', aggfunc=lambda x: (x * tx_goalies_individual.loc[x.index, 'timeOnIce']).replace(0, np.nan).sum() / tx_goalies_individual.loc[x.index, 'timeOnIce'].replace(0, np.nan).sum()),
    i_gamesPlayed=('gamesPlayed', 'sum'),
    i_gamesStarted=('gamesStarted', 'sum'),
    i_wins=('wins', 'sum'),
    i_losses = ('losses', 'sum'),
    i_otlosses = ('otLosses', 'sum'),
    i_shutouts=('shutouts', 'sum'),
    i_saves=('saves', 'sum'),
    i_shotsAgainst=('shotsAgainst', 'sum'),
    i_timeOnIce=("timeOnIce",'sum'),
).sort_values("teamAbbrev").reset_index()
tx_goalies_indiv.head()





C:\Users\kjchr\AppData\Local\Temp\ipykernel_7884\73317141.py:2: RuntimeWarning: invalid value encountered in scalar divide
  i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * tx_goalies_individual.loc[x.index, 'shotsAgainst']).replace(0, np.nan).sum() / tx_goalies_individual.loc[x.index, 'shotsAgainst'].replace(0, np.nan).sum()),


,goalieFullName,season_id,teamAbbrev,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst,i_timeOnIce
0,Kevin Boyle,20182019,ANA,0.928054,2.167236,5,4,1,3,0,1,129,139,16611
1,John Gibson,20162017,ANA,0.922364,2.236558,48,45,23,15,8,5,1200,1301,162571
2,John Gibson,20172018,ANA,0.924601,2.513007,46,46,22,15,6,1,1349,1459,157580
3,John Gibson,20182019,ANA,0.913629,2.955462,45,44,16,19,8,2,1280,1401,147388
4,John Gibson,20192020,ANA,0.903947,2.988058,48,48,19,24,5,1,1327,1468,169876


In [31]:
#last 10 games
last_10_tx = tx_goalies_individual.sort_values(['goalieFullName','game_id'], ascending=[True, False])
last_10_tx = last_10_tx.groupby("goalieFullName").head(10)

last_10_tx.head()


,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
24621,20222023,2022020863,1,1,Aaron Dell,8477180,SJS,3567,0,1,0,2.01850,25,0,0.92592,27,2
24396,20222023,2022020847,1,0,Aaron Dell,8477180,SJS,1376,0,0,0,0.00000,8,0,1.00000,8,0
25224,20222023,2022020395,1,1,Aaron Dell,8477180,SJS,3537,0,1,0,5.08905,32,0,0.86486,37,5
24549,20222023,2022020363,1,1,Aaron Dell,8477180,SJS,3477,0,1,0,2.07075,29,0,0.93548,31,2
22184,20212022,2021020744,1,1,Aaron Dell,8477180,BUF,3587,0,1,0,5.01812,42,0,0.89361,47,5


In [32]:
last_10_tx = last_10_tx.groupby(['goalieFullName', 'season_id','teamAbbrev']).agg(
    i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * tx_goalies_individual.loc[x.index, 'shotsAgainst']).sum() / tx_goalies_individual.loc[x.index, 'shotsAgainst'].sum()),
    i_GAA=pd.NamedAgg(column='goalsAgainstAverage', aggfunc=lambda x: (x * tx_goalies_individual.loc[x.index, 'timeOnIce']).sum() / tx_goalies_individual.loc[x.index, 'timeOnIce'].sum()),
    i_gamesPlayed=('gamesPlayed', 'sum'),
    i_gamesStarted=('gamesStarted', 'sum'),
    i_wins=('wins', 'sum'),
    i_losses = ('losses', 'sum'),
    i_otlosses = ('otLosses', 'sum'),
    i_shutouts=('shutouts', 'sum'),
    i_saves=('saves', 'sum'),
    i_shotsAgainst=('shotsAgainst', 'sum'),
).sort_values("teamAbbrev").reset_index()

last_10_tx.head()

C:\Users\kjchr\AppData\Local\Temp\ipykernel_7884\1841159336.py:2: RuntimeWarning: invalid value encountered in scalar divide
  i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * tx_goalies_individual.loc[x.index, 'shotsAgainst']).sum() / tx_goalies_individual.loc[x.index, 'shotsAgainst'].sum()),


,goalieFullName,season_id,teamAbbrev,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst
0,Jason LaBarbera,20142015,ANA,0.909086,2.602825,5,2,2,0,1,0,90,99
1,Chad Johnson,20182019,ANA,0.871791,3.753516,9,4,0,5,0,0,136,156
2,Lukas Dostal,20242025,ANA,0.910590,2.935400,10,8,6,2,2,0,275,302
3,Ilya Bryzgalov,20142015,ANA,0.846664,4.188372,8,6,1,4,1,0,127,150
4,Ryan Miller,20202021,ANA,0.894362,2.976842,5,4,1,2,1,0,127,142


In [33]:
goalie_games_stats_tx = tx_goalies_individual.groupby(['teamAbbrev', 'season_id']).agg(
    team_gamesPlayed=('gamesPlayed', 'sum'),
    team_gamesStarted=('gamesStarted', 'sum'),
    team_wins=('wins', 'sum'),
    team_losses=('losses', 'sum'),
    team_otlosses=('otLosses', 'sum'),
    team_shutouts=('shutouts', 'sum'),
    team_saves=('saves', 'sum'),
    team_shotsAgainst=('shotsAgainst', 'sum'),
    team_timeOnIce=("timeOnIce", 'sum'),
).reset_index()

goalie_games_stats_tx.head()

,teamAbbrev,season_id,team_gamesPlayed,team_gamesStarted,team_wins,team_losses,team_otlosses,team_shutouts,team_saves,team_shotsAgainst,team_timeOnIce
0,ANA,20132014,63,61,42,14,5,5,1571,1711,221597
1,ANA,20142015,70,63,39,17,7,4,1668,1834,230635
2,ANA,20152016,69,61,34,19,8,6,1530,1667,220642
3,ANA,20162017,73,63,32,21,10,6,1686,1840,228078
4,ANA,20172018,74,64,31,21,12,4,1989,2151,232894


In [34]:
team_goalie_stats_tx = tx_goalies_indiv.groupby(['teamAbbrev', 'season_id']).agg(
    team_savePct=pd.NamedAgg(
        column='i_savePct',
        aggfunc=lambda x: (
            (x * tx_goalies_indiv.loc[x.index, 'i_shotsAgainst']).sum()
            / tx_goalies_indiv.loc[x.index, 'i_shotsAgainst'].sum()
        )
    ),
    team_GAA=pd.NamedAgg(
        column='i_GAA',
        aggfunc=lambda x: (
            (x * tx_goalies_indiv.loc[x.index, 'i_timeOnIce']).sum()
            / tx_goalies_indiv.loc[x.index, 'i_timeOnIce'].sum()
        )
    ),
    team_wins=('i_wins', 'sum'),
    team_losses=('i_losses', 'sum'),
    team_shutouts=('i_shutouts', 'sum'),
    team_saves=('i_saves', 'sum'),
    team_shotsAgainst=('i_shotsAgainst', 'sum'),
).reset_index()

team_goalie_stats_tx = team_goalie_stats_tx.rename(columns={'teamAbbrev':'tri_code'})
team_goalie_stats_tx = team_goalie_stats_tx[['season_id', 'tri_code', 'team_savePct', 'team_GAA']]
team_goalie_stats_tx.head()



# # Export
# if team_goalie_stats_tx is not None:
#     export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\goalie_stats_agg_tx.csv"
#     os.makedirs(os.path.dirname(export_path), exist_ok=True)
#     team_goalie_stats_tx.to_csv(export_path, index=False)
#     print("✅ Final dataset exported successfully!")
# else:
#     print("❌ No data returned.")
# team_goalie_stats_tx.head()

,season_id,tri_code,team_savePct,team_GAA
0,20132014,ANA,0.918173,2.274396
1,20142015,ANA,0.909484,2.591104
2,20152016,ANA,0.917813,2.235293
3,20162017,ANA,0.916301,2.430744
4,20172018,ANA,0.924682,2.504140


In [35]:
team_goalie_stats_tx = team_goalie_stats_tx.merge(tx_teams, on = 'tri_code', how='left')
team_goalie_stats_tx.head()

,season_id,tri_code,team_savePct,team_GAA,team_id
0,20132014,ANA,0.918173,2.274396,24
1,20142015,ANA,0.909484,2.591104,24
2,20152016,ANA,0.917813,2.235293,24
3,20162017,ANA,0.916301,2.430744,24
4,20172018,ANA,0.924682,2.504140,24


### Play by Play Data

In [36]:
game_plays.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
0,20152016,2015020581,360,02:05:00,17:55:00,503,hit,258,2,16.0,90.0,-39.0,O,8476116.0,8474605.0,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,125,16,9,20152016,2015020581
1,20152016,2015020581,361,02:11:00,17:49:00,503,hit,259,2,9.0,95.0,33.0,D,8474697.0,8471254.0,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,131,16,9,20152016,2015020581
2,20152016,2015020581,362,02:23:00,17:37:00,503,hit,260,2,16.0,46.0,-40.0,O,8477929.0,8476323.0,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,143,16,9,20152016,2015020581
3,20152016,2015020581,363,02:27:00,17:33:00,516,stoppage,263,2,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,goalie-stopped-after-sog,NaN,NaN,5 on 5,in net,147,16,9,20152016,2015020581
4,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9.0,-73.0,-10.0,O,NaN,NaN,wrist,8470645.0,7.0,8.0,NaN,None,NaN,NaN,5 on 5,in net,147,16,9,20152016,2015020581


In [37]:



tx_game_plays = game_plays[
    game_plays['game_id'].isin(thanksgiving_game_ids)
]



dfs = [tx_game_plays]
exclude_cols = ["x_coord", "y_coord"]

for i, df in enumerate(dfs):
    float_cols = df.select_dtypes(include='float').columns
    float_cols = float_cols.difference(exclude_cols)
    dfs[i][float_cols] = df[float_cols].astype("Int64")

tx_game_plays.tail()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
4736614,20242025,2024020654,35,05:58:00,14:02:00,516,stoppage,374,2,<NA>,NaN,NaN,None,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,goalie-stopped-after-sog,<NA>,<NA>,5 on 5,in net,358,9,7,20242025,2024020654
4736615,20242025,2024020654,81,05:58:00,14:02:00,502,faceoff,375,2,9,-69.0,-22.0,O,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,358,9,7,20242025,2024020654
4736616,20242025,2024020654,686,06:08:00,13:52:00,503,hit,376,2,7,70.0,37.0,N,8481528,8475324,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,368,9,7,20242025,2024020654
4736617,20242025,2024020654,687,06:15:00,13:45:00,503,hit,377,2,7,-79.0,37.0,N,8478502,8482092,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,375,9,7,20242025,2024020654
4736618,20242025,2024020654,668,07:04:00,12:56:00,504,giveaway,387,2,9,-93.0,5.0,O,<NA>,<NA>,None,<NA>,<NA>,<NA>,8480801,None,<NA>,<NA>,5 on 5,in net,424,9,7,20242025,2024020654


### Rush Chances

In [38]:

tx_game_plays.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
0,20152016,2015020581,360,02:05:00,17:55:00,503,hit,258,2,16,90.0,-39.0,O,8476116,8474605,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,125,16,9,20152016,2015020581
1,20152016,2015020581,361,02:11:00,17:49:00,503,hit,259,2,9,95.0,33.0,D,8474697,8471254,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,131,16,9,20152016,2015020581
2,20152016,2015020581,362,02:23:00,17:37:00,503,hit,260,2,16,46.0,-40.0,O,8477929,8476323,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,143,16,9,20152016,2015020581
3,20152016,2015020581,363,02:27:00,17:33:00,516,stoppage,263,2,<NA>,NaN,NaN,None,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,goalie-stopped-after-sog,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581
4,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581


In [39]:
# --------------------------------------
# 1. Define event groups
# --------------------------------------
shot_events = {
    "shot-on-goal", 
    "missed-shot", 
    "goal", 
    "blocked-shot", 
    "failed-shot-attempt"
}

transition_events = {
    "giveaway",
    "takeaway",
    "hit"
}

stoppage_events = {
    "faceoff",
    "stoppage",
    "period-end",
    "game-end",
    "penalty",
    "delayed-penalty",
}

valid_prev_events = shot_events | transition_events | stoppage_events

# --------------------------------------
# 2. Filter to only events we care about
# --------------------------------------
mask = tx_game_plays["typeDescKey"].isin(valid_prev_events)
valid_plays = tx_game_plays.loc[mask].copy()

# Sort by game + time
valid_plays = valid_plays.sort_values(["game_id", "sortOrder"])

# --------------------------------------
# 3. Build "previous event" context
# --------------------------------------
grouped = valid_plays.groupby("game_id", sort=False)

valid_plays["prev_event_time"] = grouped["game_in_seconds"].shift(1)
valid_plays["prev_event_type"] = grouped["typeDescKey"].shift(1)
valid_plays["prev_zone"]       = grouped["zone_code"].shift(1)
valid_plays["prev_team_id"]    = grouped["team_id"].shift(1)

valid_plays["prev_event_was_stoppage"] = valid_plays["prev_event_type"].isin(stoppage_events)
valid_plays["time_diff"] = valid_plays["game_in_seconds"] - valid_plays["prev_event_time"]
# --------------------------------------
# 4. Rush shot condition
# --------------------------------------
is_shot = valid_plays["typeDescKey"].isin(shot_events)

# Time window: allow 0–5 seconds
quick = valid_plays["time_diff"].between(0, 5, inclusive="both")

# Possession change: previous team != shooting team
possession_change = (
    valid_plays["prev_team_id"].notna() &
    (valid_plays["prev_team_id"] != valid_plays["team_id"])
)

# Same team, but coming from DZ or NZ
same_team_dz_nz = (
    (valid_plays["prev_team_id"] == valid_plays["team_id"]) &
    (valid_plays["prev_zone"].isin(["D", "N"]))
)

no_stoppage = ~valid_plays["prev_event_was_stoppage"]

rush_mask = (
    is_shot &
    quick &
    no_stoppage &
    (possession_change | same_team_dz_nz)
)

tx_rush_df = valid_plays.loc[rush_mask].copy()


tx_rush_df["rush_shot"] = 1


In [40]:
# --------------------------------------
# 5. Rush shots FOR (team taking the rush shot)
# --------------------------------------
tx_rush_shots_for = (
    tx_rush_df
    .groupby(["season_id", "team_id"])
    .size()
    .reset_index(name="rush_shots_for")
)

# --------------------------------------
# 6. Rush shots AGAINST (team whose previous event enabled the rush)
# --------------------------------------
tx_rush_shots_against = (
    tx_rush_df
    .groupby(["season_id", "prev_team_id"])
    .size()
    .reset_index(name="rush_shots_against")
    .rename(columns={"prev_team_id": "team_id"})
)

# --------------------------------------
# 7. Merge FOR + AGAINST
# --------------------------------------
tx_rush_stats = tx_rush_shots_for.merge(
    tx_rush_shots_against,
    on=["season_id", "team_id"],
    how="outer"
)

tx_rush_stats = tx_rush_stats.fillna(0)

tx_rush_stats["rush_shots_for"] = tx_rush_stats["rush_shots_for"].astype(int)
tx_rush_stats["rush_shots_against"] = tx_rush_stats["rush_shots_against"].astype(int)

tx_rush_stats["rush_shot_differential"] = (
    tx_rush_stats["rush_shots_for"] - tx_rush_stats["rush_shots_against"]
)

tx_rush_stats = (
    tx_rush_stats
    .sort_values(["season_id", "rush_shot_differential"], ascending=[True, False])
    .reset_index(drop=True)
)

tx_rush_stats.head()


,season_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential
0,20132014,16,392,319,73
1,20132014,28,404,332,72
2,20132014,1,271,211,60
3,20132014,23,352,310,42
4,20132014,13,311,274,37


In [41]:
tx_rush_stats = tx_rush_stats.merge(tx_teams, on = "team_id", how="left")
tx_rush_df_abbrev = tx_rush_df[['game_id', 'eventId', 'game_in_seconds', 'prev_event_time', 'prev_zone', 'prev_event_type', 'prev_team_id', 'prev_event_was_stoppage', 'time_diff', 'rush_shot']]


In [42]:
tx_rush_df_abbrev.head()

,game_id,eventId,game_in_seconds,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
22533,2013020001,55,105,100.0,O,hit,8,False,5.0,1
22550,2013020001,63,178,176.0,D,hit,8,False,2.0,1
22565,2013020001,71,225,224.0,D,giveaway,10,False,1.0,1
22577,2013020001,78,320,316.0,D,blocked-shot,10,False,4.0,1
22608,2013020001,97,692,689.0,D,hit,10,False,3.0,1


### All Shot Attempts

In [43]:
tx_all_shots_made = tx_game_plays.copy()
tx_all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
0,20152016,2015020581,360,02:05:00,17:55:00,503,hit,258,2,16,90.0,-39.0,O,8476116,8474605,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,125,16,9,20152016,2015020581
1,20152016,2015020581,361,02:11:00,17:49:00,503,hit,259,2,9,95.0,33.0,D,8474697,8471254,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,131,16,9,20152016,2015020581
2,20152016,2015020581,362,02:23:00,17:37:00,503,hit,260,2,16,46.0,-40.0,O,8477929,8476323,None,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,5 on 5,in net,143,16,9,20152016,2015020581
3,20152016,2015020581,363,02:27:00,17:33:00,516,stoppage,263,2,<NA>,NaN,NaN,None,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,goalie-stopped-after-sog,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581
4,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581


In [44]:
tx_all_shots_made = tx_all_shots_made[tx_all_shots_made["typeCode"].isin([505, 506, 507, 508, 537])]
tx_all_shots_made["goalie"] = tx_all_shots_made["goalie"].fillna(0)
tx_all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
4,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581
6,20152016,2015020581,222,03:26:00,16:34:00,506,shot-on-goal,274,2,9,-69.0,27.0,O,<NA>,<NA>,wrist,8470645,8,8,<NA>,None,<NA>,<NA>,5 on 5,in net,206,16,9,20152016,2015020581
9,20152016,2015020581,224,03:32:00,16:28:00,508,blocked-shot,278,2,9,-46.0,2.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8473604,<NA>,5 on 5,in net,212,16,9,20152016,2015020581
13,20152016,2015020581,226,03:48:00,16:12:00,506,shot-on-goal,282,2,9,-35.0,18.0,O,<NA>,<NA>,slap,8470645,9,8,<NA>,None,<NA>,<NA>,5 on 5,in net,228,16,9,20152016,2015020581
17,20152016,2015020581,228,04:39:00,15:21:00,507,missed-shot,293,2,9,-45.0,12.0,O,<NA>,<NA>,wrist,8470645,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,279,16,9,20152016,2015020581


In [45]:
#add in the rush shot attempts
tx_all_shots_made = tx_all_shots_made.merge(tx_rush_df_abbrev, on=['game_id', 'eventId','game_in_seconds'], how = 'left')
tx_all_shots_made['rush_shot']= tx_all_shots_made['rush_shot'].fillna(0)
tx_all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
0,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
1,20152016,2015020581,222,03:26:00,16:34:00,506,shot-on-goal,274,2,9,-69.0,27.0,O,<NA>,<NA>,wrist,8470645,8,8,<NA>,None,<NA>,<NA>,5 on 5,in net,206,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
2,20152016,2015020581,224,03:32:00,16:28:00,508,blocked-shot,278,2,9,-46.0,2.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8473604,<NA>,5 on 5,in net,212,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
3,20152016,2015020581,226,03:48:00,16:12:00,506,shot-on-goal,282,2,9,-35.0,18.0,O,<NA>,<NA>,slap,8470645,9,8,<NA>,None,<NA>,<NA>,5 on 5,in net,228,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0
4,20152016,2015020581,228,04:39:00,15:21:00,507,missed-shot,293,2,9,-45.0,12.0,O,<NA>,<NA>,wrist,8470645,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,279,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0


In [46]:
# Identify Corsi shots - those only in the O Zone

tx_all_shots_made['corsi'] = np.where(tx_all_shots_made['zone_code']== "O", 1, 0)
tx_all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi
0,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1
1,20152016,2015020581,222,03:26:00,16:34:00,506,shot-on-goal,274,2,9,-69.0,27.0,O,<NA>,<NA>,wrist,8470645,8,8,<NA>,None,<NA>,<NA>,5 on 5,in net,206,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1
2,20152016,2015020581,224,03:32:00,16:28:00,508,blocked-shot,278,2,9,-46.0,2.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8473604,<NA>,5 on 5,in net,212,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0
3,20152016,2015020581,226,03:48:00,16:12:00,506,shot-on-goal,282,2,9,-35.0,18.0,O,<NA>,<NA>,slap,8470645,9,8,<NA>,None,<NA>,<NA>,5 on 5,in net,228,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1
4,20152016,2015020581,228,04:39:00,15:21:00,507,missed-shot,293,2,9,-45.0,12.0,O,<NA>,<NA>,wrist,8470645,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,279,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1


### SH%
- SH%: Percentage of Shots for that team that were Goals. GF*100/SF
- SV%: Percentage of Shots against that team that were not Goals. 100-(GA*100/SA)

In [47]:
tx_sh_percent = tx_all_shots_made[['game_id', 'season_id', 'team_id', 'typeDescKey']].copy()

# Create shot-type indicator columns
tx_sh_percent['block'] = np.where(tx_sh_percent['typeDescKey'] == 'blocked-shot', 1, 0)
tx_sh_percent['goal'] = np.where(tx_sh_percent['typeDescKey'] == 'goal', 1, 0)

# A "shot" is any attempt that is NOT blocked
# (blocked shots never reach the net, so they do not count toward SOG)
tx_sh_percent['shots'] = np.where(tx_sh_percent['block'] == 1, 0, 1)

# Group by season + team
tx_sh_percent = (
    tx_sh_percent
        .groupby(['season_id', 'team_id'], as_index=False)
        .agg(
            shots=('shots', 'sum'),
            goals=('goal', 'sum'),
        )
)

# Safe shooting percentage (avoids division-by-zero)
tx_sh_percent['shot_pct'] = (
    tx_sh_percent.apply(
        lambda row: round((row['goals'] * 100) / row['shots'], 2)
        if row['shots'] > 0 else 0,
        axis=1
    )
)


tx_sh_percent.head()

,season_id,team_id,shots,goals,shot_pct
0,20132014,1,2230,141,6.32
1,20132014,2,2595,176,6.78
2,20132014,3,2705,162,5.99
3,20132014,4,2512,172,6.85
4,20132014,5,2473,193,7.80


In [48]:
tx_shots_against = tx_all_shots_made[['game_id','season_id', 'team_id', 'typeDescKey']].copy()

# Set up columns
tx_shots_against['block'] = np.where(tx_shots_against['typeDescKey'] == 'blocked-shot', 1, 0)
tx_shots_against['goal'] = np.where(tx_shots_against['typeDescKey'] == "goal", 1, 0)
tx_shots_against['shot'] = np.where(tx_shots_against['typeDescKey'] != "goal", 1, 0)
tx_shots_against['shots'] = np.where(tx_shots_against['block'] == 1, 0, 1)


# ----------------------------------------------------
# FIXED OPPONENT MAP (NO APPLY/STACK ERRORS)
# ----------------------------------------------------

# Extract the two team_ids per game
tx_team_pairs = (
    tx_shots_against[['game_id', 'team_id']]
    .drop_duplicates()
    .groupby('game_id')['team_id']
    .apply(list)
)

# Build opponent mapping as a dict-of-dicts
opponent_map = {}

for game_id, ids in tx_team_pairs.items():
    if len(ids) == 2:
        opponent_map[game_id] = {
            ids[0]: ids[1],
            ids[1]: ids[0]
        }
    else:
        # fallback in case of weird data
        opponent_map[game_id] = {}


# Assign opponent team for each shot attempt
def get_opponent(row):
    lookup = opponent_map.get(row['game_id'], {})
    return lookup.get(row['team_id'])

tx_shots_against['opponent_team_id'] = tx_shots_against.apply(get_opponent, axis=1)

tx_shots_against.head()

,game_id,season_id,team_id,typeDescKey,block,goal,shot,shots,opponent_team_id
0,2015020581,20152016,9,shot-on-goal,0,0,1,1,16
1,2015020581,20152016,9,shot-on-goal,0,0,1,1,16
2,2015020581,20152016,9,blocked-shot,1,0,1,0,16
3,2015020581,20152016,9,shot-on-goal,0,0,1,1,16
4,2015020581,20152016,9,missed-shot,0,0,1,1,16


In [49]:

# ===============================
# Opponent / Against Shot Metrics
# ===============================

# Group by season + opponent team ID
tx_against = (
    tx_shots_against
        .groupby(['season_id', 'opponent_team_id'], as_index=False)
        .agg(
            shots_against=('shots', 'sum'),
            goals_against=('goal', 'sum')
        )
)

# Rename opponent_team_id → team_id (so it's consistent with your "for" dataset)
tx_against = tx_against.rename(columns={'opponent_team_id': 'team_id'})

# Safe shot percentage against
# (prevents division-by-zero warnings if a team allows 0 shots)
tx_against['shot_pct_against'] = (
    tx_against.apply(
        lambda row: round((row['goals_against'] * 100) / row['shots_against'], 2)
        if row['shots_against'] > 0 else 0,
        axis=1
    )
)

# Preview
tx_against.head()

,season_id,team_id,shots_against,goals_against,shot_pct_against
0,20132014,1,2075,151,7.28
1,20132014,2,2619,211,8.06
2,20132014,3,2468,153,6.20
3,20132014,4,2522,181,7.18
4,20132014,5,2365,145,6.13


In [50]:

# Merge both DataFrames on team_id
tx_shooting_stats = tx_sh_percent.merge(tx_against, on=['team_id','season_id'], how='left')

# Calculate Save Percentage (SV%)
tx_shooting_stats['sv_pct'] = 100 - ((tx_shooting_stats['goals_against'] * 100) / tx_shooting_stats['shots_against'])

# Optional: round the SV% column
tx_shooting_stats['sv_pct'] = tx_shooting_stats['sv_pct'].round(2)

# Done!
tx_shooting_stats.head()


,season_id,team_id,shots,goals,shot_pct,shots_against,goals_against,shot_pct_against,sv_pct
0,20132014,1,2230,141,6.32,2075,151,7.28,92.72
1,20132014,2,2595,176,6.78,2619,211,8.06,91.94
2,20132014,3,2705,162,5.99,2468,153,6.20,93.80
3,20132014,4,2512,172,6.85,2522,181,7.18,92.82
4,20132014,5,2473,193,7.80,2365,145,6.13,93.87


In [51]:
#PDO - Shooting percentage plus save percentage. (GF/SF)+(GA/SA)
tx_shooting_stats['pdo'] = tx_shooting_stats['shot_pct'] + tx_shooting_stats['sv_pct']
tx_shooting_stats = tx_shooting_stats[['team_id','season_id', 'shot_pct', 'shots', 'shots_against', 'shot_pct_against', 'sv_pct', 'pdo']]
tx_shooting_stats.head()

,team_id,season_id,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo
0,1,20132014,6.32,2230,2075,7.28,92.72,99.04
1,2,20132014,6.78,2595,2619,8.06,91.94,98.72
2,3,20132014,5.99,2705,2468,6.20,93.80,99.79
3,4,20132014,6.85,2512,2522,7.18,92.82,99.67
4,5,20132014,7.80,2473,2365,6.13,93.87,101.67


### Danger Zones

In [52]:
# Danger zone

# Define conditions based on the corrected MD and HD coordinates

# High Danger Zone (HD) - Right side of the rink
hd_condition_right = (
    (tx_all_shots_made["x_coord"] >= 66.5)
    & (tx_all_shots_made["x_coord"] <= 89)
    & (tx_all_shots_made["y_coord"] >= -7)
    & (tx_all_shots_made["y_coord"] <= 7)
)

# High Danger Zone (HD) - Left side of the rink (mirrored)
hd_condition_left = (
    (tx_all_shots_made["x_coord"] >= -89)
    & (tx_all_shots_made["x_coord"] <= -66.5)
    & (tx_all_shots_made["y_coord"] >= -7)
    & (tx_all_shots_made["y_coord"] <= 7)
)

# Medium Danger Zone (MD) - Right side
md_condition_right = (
    (
        (tx_all_shots_made["x_coord"] >= 69)
        & (tx_all_shots_made["x_coord"] <= 89)
        & (tx_all_shots_made["y_coord"] >= -22)
        & (tx_all_shots_made["y_coord"] <= -7)
    )
    | (
        (tx_all_shots_made["x_coord"] >= 54)
        & (tx_all_shots_made["x_coord"] <= 69)
        & (tx_all_shots_made["y_coord"] >= -22)
        & (tx_all_shots_made["y_coord"] <= 22)
    )
    | (
        (tx_all_shots_made["x_coord"] >= 40)
        & (tx_all_shots_made["x_coord"] <= 54)
        & (tx_all_shots_made["y_coord"] >= -7)
        & (tx_all_shots_made["y_coord"] <= 7)
    )
)

# Medium Danger Zone (MD) - Left side (mirrored)
md_condition_left = (
    (
        (tx_all_shots_made["x_coord"] >= -89)
        & (tx_all_shots_made["x_coord"] <= -69)
        & (tx_all_shots_made["y_coord"] >= -22)
        & (tx_all_shots_made["y_coord"] <= -7)
    )
    | (
        (tx_all_shots_made["x_coord"] >= -69)
        & (tx_all_shots_made["x_coord"] <= -54)
        & (tx_all_shots_made["y_coord"] >= -22)
        & (tx_all_shots_made["y_coord"] <= 22)
    )
    | (
        (tx_all_shots_made["x_coord"] >= -54)
        & (tx_all_shots_made["x_coord"] <= -40)
        & (tx_all_shots_made["y_coord"] >= -7)
        & (tx_all_shots_made["y_coord"] <= 7)
    )
)

# Low Danger Zone (LD) - Every other coordinate
ld_condition = ~(
    hd_condition_right | hd_condition_left | md_condition_right | md_condition_left
)

# Apply conditions to classify shots
conditions = [
    hd_condition_right | hd_condition_left,
    md_condition_right | md_condition_left,
    ld_condition,
]
values = ["HD", "MD", "LD"]

tx_all_shots_made["shot_danger"] = np.select(conditions, values, default="LD")

# Create new columns for HD, MD, and LD shots
tx_all_shots_made["hd_shot"] = (tx_all_shots_made["shot_danger"] == "HD").astype(int)
tx_all_shots_made["md_shot"] = (tx_all_shots_made["shot_danger"] == "MD").astype(int)
tx_all_shots_made["ld_shot"] = (tx_all_shots_made["shot_danger"] == "LD").astype(int)


In [53]:
tx_all_shots_made["game_time"] = (
    tx_all_shots_made["period_number"] - 1
) * 1200 + tx_all_shots_made["game_in_seconds"]

# Sort by game_id and game_time
# all_shots_made = all_shots_made.sort_values(by=['game_id', 'game_time'])

# Calculate time difference from the last shot in the same game
tx_all_shots_made["time_since_last_shot"] = tx_all_shots_made.groupby("game_id")[
    "game_time"
].diff()

# Define a rebound as a shot that follows another shot within 3 seconds
tx_all_shots_made["is_rebound"] = (tx_all_shots_made["time_since_last_shot"] <= 3).astype(int)

tx_all_shots_made.head()


,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound
0,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,1347,NaN,0
1,20152016,2015020581,222,03:26:00,16:34:00,506,shot-on-goal,274,2,9,-69.0,27.0,O,<NA>,<NA>,wrist,8470645,8,8,<NA>,None,<NA>,<NA>,5 on 5,in net,206,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,1406,59.0,0
2,20152016,2015020581,224,03:32:00,16:28:00,508,blocked-shot,278,2,9,-46.0,2.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8473604,<NA>,5 on 5,in net,212,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0,MD,0,1,0,1412,6.0,0
3,20152016,2015020581,226,03:48:00,16:12:00,506,shot-on-goal,282,2,9,-35.0,18.0,O,<NA>,<NA>,slap,8470645,9,8,<NA>,None,<NA>,<NA>,5 on 5,in net,228,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,1428,16.0,0
4,20152016,2015020581,228,04:39:00,15:21:00,507,missed-shot,293,2,9,-45.0,12.0,O,<NA>,<NA>,wrist,8470645,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,279,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,1479,51.0,0


### Scoring Chances

In [54]:

# Define conditions
conditions = [
    tx_all_shots_made['hd_shot'] == 1,  # High Danger Shot
    tx_all_shots_made['md_shot'] == 1,  # Medium Danger Shot
    tx_all_shots_made['ld_shot'] == 1   # Low Danger Shot
]

# Define corresponding values
values = [3, 2, 1]  # Base values for scoring chances

# Apply np.select() to determine base scoring chance
tx_all_shots_made['scoring_chance'] = np.select(conditions, values, default=1)

# Apply additional logic: If rush shot or rebound, increase scoring chance by 1
tx_all_shots_made['scoring_chance'] += (tx_all_shots_made['rush_shot'] == 1) | (tx_all_shots_made['is_rebound'] == 1)
tx_all_shots_made['scoring_chance'] -= tx_all_shots_made['typeCode'] == 508

tx_all_shots_made['scoring_chance'] = np.where(tx_all_shots_made['scoring_chance'] >=2, 1, 0)
# Display result
tx_all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound,scoring_chance
0,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,1347,NaN,0,1
1,20152016,2015020581,222,03:26:00,16:34:00,506,shot-on-goal,274,2,9,-69.0,27.0,O,<NA>,<NA>,wrist,8470645,8,8,<NA>,None,<NA>,<NA>,5 on 5,in net,206,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,1406,59.0,0,0
2,20152016,2015020581,224,03:32:00,16:28:00,508,blocked-shot,278,2,9,-46.0,2.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8473604,<NA>,5 on 5,in net,212,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0,MD,0,1,0,1412,6.0,0,0
3,20152016,2015020581,226,03:48:00,16:12:00,506,shot-on-goal,282,2,9,-35.0,18.0,O,<NA>,<NA>,slap,8470645,9,8,<NA>,None,<NA>,<NA>,5 on 5,in net,228,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,1428,16.0,0,0
4,20152016,2015020581,228,04:39:00,15:21:00,507,missed-shot,293,2,9,-45.0,12.0,O,<NA>,<NA>,wrist,8470645,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,279,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,LD,0,0,1,1479,51.0,0,0


In [55]:
tx_scoring_chance_stats = tx_all_shots_made[tx_all_shots_made['scoring_chance']==1]
tx_scoring_chance_stats.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound,scoring_chance
0,20152016,2015020581,219,02:27:00,17:33:00,506,shot-on-goal,264,2,9,-73.0,-10.0,O,<NA>,<NA>,wrist,8470645,7,8,<NA>,None,<NA>,<NA>,5 on 5,in net,147,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,1347,NaN,0,1
5,20152016,2015020581,229,06:08:00,13:52:00,507,missed-shot,302,2,16,52.0,-16.0,O,<NA>,<NA>,wrist,8467950,<NA>,<NA>,<NA>,wide-of-net,<NA>,<NA>,5 on 5,in net,368,16,9,20152016,2015020581,364.0,D,hit,9,False,4.0,1.0,1,LD,0,0,1,1568,89.0,0,1
6,20152016,2015020581,230,06:15:00,13:45:00,508,blocked-shot,304,2,16,67.0,5.0,D,<NA>,<NA>,None,0,<NA>,<NA>,<NA>,None,8474578,<NA>,5 on 5,in net,375,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,0,HD,1,0,0,1575,7.0,0,1
10,20152016,2015020581,234,07:51:00,12:09:00,506,shot-on-goal,327,2,16,57.0,-9.0,O,<NA>,<NA>,wrist,8467950,9,10,<NA>,None,<NA>,<NA>,5 on 5,in net,471,16,9,20152016,2015020581,NaN,NaN,NaN,<NA>,NaN,NaN,0.0,1,MD,0,1,0,1671,52.0,0,1
11,20152016,2015020581,235,08:13:00,11:47:00,505,goal,331,2,16,37.0,16.0,O,<NA>,<NA>,slap,8467950,<NA>,<NA>,<NA>,None,<NA>,8470281,5 on 5,in net,493,16,9,20152016,2015020581,488.0,D,hit,9,False,5.0,1.0,1,LD,0,0,1,1693,22.0,0,1


In [56]:

tx_scoring_chance_stats = tx_scoring_chance_stats[['game_id', 'season_id','eventId', 'timeInPeriod', 'timeRemaining',  'team_id','typeCode', 'typeDescKey','shot_type', 'goalie',
        'situation', 'goalie_situation',
       'game_in_seconds', 'home_team', 'away_team', 'away_sog', 'home_sog','rush_shot', 'corsi',
       'hd_shot', 'md_shot', 'ld_shot', 'scoring_chance']]
tx_scoring_chance_stats.head()

,game_id,season_id,eventId,timeInPeriod,timeRemaining,team_id,typeCode,typeDescKey,shot_type,goalie,situation,goalie_situation,game_in_seconds,home_team,away_team,away_sog,home_sog,rush_shot,corsi,hd_shot,md_shot,ld_shot,scoring_chance
0,2015020581,20152016,219,02:27:00,17:33:00,9,506,shot-on-goal,wrist,8470645,5 on 5,in net,147,16,9,7,8,0.0,1,0,1,0,1
5,2015020581,20152016,229,06:08:00,13:52:00,16,507,missed-shot,wrist,8467950,5 on 5,in net,368,16,9,<NA>,<NA>,1.0,1,0,0,1,1
6,2015020581,20152016,230,06:15:00,13:45:00,16,508,blocked-shot,None,0,5 on 5,in net,375,16,9,<NA>,<NA>,0.0,0,1,0,0,1
10,2015020581,20152016,234,07:51:00,12:09:00,16,506,shot-on-goal,wrist,8467950,5 on 5,in net,471,16,9,9,10,0.0,1,0,1,0,1
11,2015020581,20152016,235,08:13:00,11:47:00,16,505,goal,slap,8467950,5 on 5,in net,493,16,9,<NA>,<NA>,1.0,1,0,0,1,1


In [57]:
tx_scoring_chance_stats['shooting_team_id'] = tx_scoring_chance_stats['team_id']

# Identify blocked shots (typeCode = 508)
mask = tx_scoring_chance_stats['typeCode'] == 508

# For blocked shots, shooter is the OTHER team
tx_home_block = (tx_scoring_chance_stats['team_id'] == tx_scoring_chance_stats['home_team'])
tx_away_block = (tx_scoring_chance_stats['team_id'] == tx_scoring_chance_stats['away_team'])

# Shooter when home player blocked → shooter = away_team
tx_scoring_chance_stats.loc[mask, 'shooting_team_id'] = tx_scoring_chance_stats['away_team'].where(
    tx_home_block, tx_scoring_chance_stats['shooting_team_id']
)

# Shooter when away player blocked → shooter = home_team
tx_scoring_chance_stats.loc[mask, 'shooting_team_id'] = tx_scoring_chance_stats['home_team'].where(
    tx_away_block, tx_scoring_chance_stats['shooting_team_id']
)


tx_scoring_chance_stats['scoring_chance_shot'] = np.where(
    tx_scoring_chance_stats['typeCode'].isin([505, 506]),
    1, 
    0
)

tx_scoring_chance_stats['scoring_chance_goal'] = np.where(
    tx_scoring_chance_stats['typeCode'] == 505, 
    1, 
    0
)


tx_scoring_chance_stats.head()

,game_id,season_id,eventId,timeInPeriod,timeRemaining,team_id,typeCode,typeDescKey,shot_type,goalie,situation,goalie_situation,game_in_seconds,home_team,away_team,away_sog,home_sog,rush_shot,corsi,hd_shot,md_shot,ld_shot,scoring_chance,shooting_team_id,scoring_chance_shot,scoring_chance_goal
0,2015020581,20152016,219,02:27:00,17:33:00,9,506,shot-on-goal,wrist,8470645,5 on 5,in net,147,16,9,7,8,0.0,1,0,1,0,1,9,1,0
5,2015020581,20152016,229,06:08:00,13:52:00,16,507,missed-shot,wrist,8467950,5 on 5,in net,368,16,9,<NA>,<NA>,1.0,1,0,0,1,1,16,0,0
6,2015020581,20152016,230,06:15:00,13:45:00,16,508,blocked-shot,None,0,5 on 5,in net,375,16,9,<NA>,<NA>,0.0,0,1,0,0,1,9,0,0
10,2015020581,20152016,234,07:51:00,12:09:00,16,506,shot-on-goal,wrist,8467950,5 on 5,in net,471,16,9,9,10,0.0,1,0,1,0,1,16,1,0
11,2015020581,20152016,235,08:13:00,11:47:00,16,505,goal,slap,8467950,5 on 5,in net,493,16,9,<NA>,<NA>,1.0,1,0,0,1,1,16,1,1


In [58]:

# Make sure season_id is included
tx_scf_stats = tx_scoring_chance_stats[['season_id', 'shooting_team_id',
                                  'scoring_chance', 'scoring_chance_shot',
                                  'scoring_chance_goal']].copy()

# Group by season and team
tx_scf_stats = (
    tx_scf_stats
        .groupby(['season_id', 'shooting_team_id'], as_index=False)
        .agg(
            SCF=('scoring_chance', 'sum'),
            SCSF=('scoring_chance_shot', 'sum'),
            SCGF=('scoring_chance_goal', 'sum'),
        )
        .rename(columns={'shooting_team_id': 'team_id'})
)

# Preview
tx_scf_stats.head()

,season_id,team_id,SCF,SCSF,SCGF
0,20132014,1,1422,880,109
1,20132014,2,1833,1169,146
2,20132014,3,1738,1111,135
3,20132014,4,1546,923,142
4,20132014,5,1560,979,156


In [59]:

# ============================================================
# Scoring Chances Against (SCA) by Season + Team
# ============================================================

# --- Home SCA ---
tx_sca_home = (
    tx_scoring_chance_stats
        .groupby(['season_id', 'home_team'], as_index=False)
        .agg(SCA=('scoring_chance', 'sum'))
        .rename(columns={'home_team': 'team_id'})
)

# --- Away SCA ---
tx_sca_away = (
    tx_scoring_chance_stats
        .groupby(['season_id', 'away_team'], as_index=False)
        .agg(SCA=('scoring_chance', 'sum'))
        .rename(columns={'away_team': 'team_id'})
)

# --- Combine home + away ---
tx_sca_stats = (
    pd.concat([tx_sca_home, tx_sca_away], ignore_index=True)
      .groupby(['season_id', 'team_id'], as_index=False)
      .agg(SCA=('SCA', 'sum'))
)

# --- Merge SCF + SCA ---
tx_final_scoring_stats = (
    tx_scf_stats.merge(tx_sca_stats, on=['season_id', 'team_id'], how='left')
)

tx_final_scoring_stats.head()

,season_id,team_id,SCF,SCSF,SCGF,SCA
0,20132014,1,1422,880,109,2755.0
1,20132014,2,1833,1169,146,3684.0
2,20132014,3,1738,1111,135,3392.0
3,20132014,4,1546,923,142,3103.0
4,20132014,5,1560,979,156,2990.0


### Strength of Schedule

In [60]:

# ===============================================================
# STEP 1 — Compute Win% per season_id + team
# ===============================================================

tx_team_stats = []

def calculate_team_win_percentage(season_df, team_abbrev):
    # Wins (home or away)
    wins = (
        len(season_df[(season_df["awayTeam.abbrev"] == team_abbrev) &
                      (season_df["awayTeam.score"] > season_df["homeTeam.score"])])
        +
        len(season_df[(season_df["homeTeam.abbrev"] == team_abbrev) &
                      (season_df["homeTeam.score"] > season_df["awayTeam.score"])])
    )

    # Total games
    games = len(season_df[(season_df["awayTeam.abbrev"] == team_abbrev) |
                          (season_df["homeTeam.abbrev"] == team_abbrev)])

    return wins / games if games > 0 else 0


# Loop by season_id
for season_id, season_df in thanksgiving_game_summary.groupby("season_id"):
    team_abbrevs = pd.concat([
        season_df["awayTeam.abbrev"],
        season_df["homeTeam.abbrev"]
    ]).unique()

    for team_abbrev in team_abbrevs:
        win_pct = calculate_team_win_percentage(season_df, team_abbrev)

        tx_team_stats.append({
            "season_id": season_id,
            "tri_code": team_abbrev,
            "win_percentage": win_pct
        })

tx_team_stats_df = pd.DataFrame(tx_team_stats)

# ===============================================================
# STEP 2 — Map Win% back into the main games dataframe
# ===============================================================

thanksgiving_game_summary = thanksgiving_game_summary.merge(
    tx_team_stats_df.rename(columns={"tri_code": "awayTeam.abbrev"}),
    on=["season_id", "awayTeam.abbrev"],
    how="left"
).rename(columns={"win_percentage": "awayTeam_win_percentage"})

thanksgiving_game_summary = thanksgiving_game_summary.merge(
    tx_team_stats_df.rename(columns={"tri_code": "homeTeam.abbrev"}),
    on=["season_id", "homeTeam.abbrev"],
    how="left"
).rename(columns={"win_percentage": "homeTeam_win_percentage"})


# ===============================================================
# STEP 3 — Calculate SOS per season_id + team_id
# ===============================================================

# Away Team SOS = average win% of opponents (home teams)
tx_away_team_sos = (
    thanksgiving_game_summary.groupby(["season_id", "awayTeam.abbrev"])["homeTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"awayTeam.abbrev": "tri_code",
                     "homeTeam_win_percentage": "SOS"})
)

# Home Team SOS = average win% of opponents (away teams)
tx_home_team_sos = (
    thanksgiving_game_summary.groupby(["season_id", "homeTeam.abbrev"])["awayTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"homeTeam.abbrev": "tri_code",
                     "awayTeam_win_percentage": "SOS"})
)

# Combine home + away appearances
tx_team_sos = pd.concat([tx_away_team_sos, tx_home_team_sos], ignore_index=True)

# Final SOS per season + team
tx_team_sos = (
    tx_team_sos.groupby(["season_id", "tri_code"], as_index=False)["SOS"]
    .mean()
)

# Final output
tx_team_sos.head()


,season_id,tri_code,SOS
0,20132014,ANA,0.494596
1,20132014,BOS,0.484655
2,20132014,BUF,0.518278
3,20132014,CAR,0.498154
4,20132014,CBJ,0.498105


In [61]:
tx_team_stats_df.head()

,season_id,tri_code,win_percentage
0,20132014,TOR,0.524590
1,20132014,WSH,0.466667
2,20132014,WPG,0.475410
3,20132014,BUF,0.300000
4,20132014,ANA,0.688525


In [62]:
tx_team_stats_df.columns.tolist()


['season_id', 'tri_code', 'win_percentage']

In [63]:
# Step 1 — Filter games AFTER Thanksgiving
tx_future_games = history_summary[
    (~history_summary["game_id"].isin(thanksgiving_game_ids)) &
    (history_summary["season_id"] != 20252026)
]


# ======================================================
# Step 2 — Map win % by (season_id, team_abbrev)
# ======================================================
# Step 2 — Map win % by (season_id, tri_code)

# Build lookup keyed by (season_id, team_abbrev aka tri_code)
lookup = tx_team_stats_df.set_index(["season_id", "tri_code"])["win_percentage"]

# Build key pairs for mapping:
away_keys = list(zip(tx_future_games["season_id"], tx_future_games["awayTeam.abbrev"]))
home_keys = list(zip(tx_future_games["season_id"], tx_future_games["homeTeam.abbrev"]))

# Apply mapping:
tx_future_games["awayTeam_win_percentage"] = [lookup.get(k, np.nan) for k in away_keys]
tx_future_games["homeTeam_win_percentage"] = [lookup.get(k, np.nan) for k in home_keys]

# ======================================================
# Step 3 — Compute Remaining SOS for home + away teams
# ======================================================

# Away team SOS (they are playing the home's win%)
tx_away_team_remaining_sos = (
    tx_future_games.groupby(["season_id", "awayTeam.abbrev"])["homeTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"awayTeam.abbrev": "tri_code",
                     "homeTeam_win_percentage": "Remaining_SOS"})
)

# Home team SOS (they are playing the away's win%)
tx_home_team_remaining_sos = (
    tx_future_games.groupby(["season_id", "homeTeam.abbrev"])["awayTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"homeTeam.abbrev": "tri_code",
                     "awayTeam_win_percentage": "Remaining_SOS"})
)


# ======================================================
# Step 4 — Combine home + away and final aggregation
# ======================================================

tx_remaining_team_sos = pd.concat(
    [tx_away_team_remaining_sos, tx_home_team_remaining_sos],
    axis=0,
    ignore_index=True
)

# Final average remaining SOS per team per season
tx_remaining_team_sos = (
    tx_remaining_team_sos
    .groupby(["season_id", "tri_code"], as_index=False)["Remaining_SOS"]
    .mean()
    .sort_values(["season_id", "tri_code"])
)

tx_remaining_team_sos.head()


,season_id,tri_code,Remaining_SOS
0,20132014,ANA,0.482906
1,20132014,BOS,0.479566
2,20132014,BUF,0.479450
3,20132014,CAR,0.485708
4,20132014,CBJ,0.497083


In [64]:
tx_strength = tx_team_sos.merge(tx_remaining_team_sos, on=['tri_code','season_id'], how = 'inner')
tx_strength = tx_strength[['season_id', 'tri_code', 'SOS', 'Remaining_SOS']]
tx_strength = tx_strength.merge(tx_teams, on='tri_code', how='left')
tx_strength.head()

,season_id,tri_code,SOS,Remaining_SOS,team_id
0,20132014,ANA,0.494596,0.482906,24
1,20132014,BOS,0.484655,0.479566,6
2,20132014,BUF,0.518278,0.479450,7
3,20132014,CAR,0.498154,0.485708,12
4,20132014,CBJ,0.498105,0.497083,29


In [65]:

# --------------------------------------------------------------
# Step 3: Calculate Remaining SOS for Away Teams
# --------------------------------------------------------------

tx_away_team_remaining_sos = (
    tx_future_games.groupby(["season_id", "awayTeam.abbrev"])["homeTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={
        "awayTeam.abbrev": "tri_code",
        "homeTeam_win_percentage": "Remaining_SOS"
    })
)

# --------------------------------------------------------------
# Step 3b: Calculate Remaining SOS for Home Teams
# --------------------------------------------------------------

tx_home_team_remaining_sos = (
    tx_future_games.groupby(["season_id", "homeTeam.abbrev"])["awayTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={
        "homeTeam.abbrev": "tri_code",
        "awayTeam_win_percentage": "Remaining_SOS"
    })
)

# --------------------------------------------------------------
# Step 4: Combine Home + Away SOS rows
# --------------------------------------------------------------

tx_remaining_team_sos = pd.concat(
    [tx_away_team_remaining_sos, tx_home_team_remaining_sos],
    axis=0,
    ignore_index=True
)

# --------------------------------------------------------------
# Step 5: Final aggregation — group by season + team code
# --------------------------------------------------------------

tx_remaining_team_sos = (
    tx_remaining_team_sos
    .groupby(["season_id", "tri_code"], as_index=False)["Remaining_SOS"]
    .mean()
)

tx_remaining_team_sos.head()


,season_id,tri_code,Remaining_SOS
0,20132014,ANA,0.482906
1,20132014,BOS,0.479566
2,20132014,BUF,0.479450
3,20132014,CAR,0.485708
4,20132014,CBJ,0.497083


In [66]:
tx_team_sos = tx_team_sos.rename(columns={'awayTeam.abbrev':'tri_code'})
tx_strength = tx_team_sos.merge(tx_remaining_team_sos, on=['season_id','tri_code'], how = 'inner')
tx_strength = tx_strength.merge(tx_teams, on='tri_code', how='left')
tx_strength.head()

,season_id,tri_code,SOS,Remaining_SOS,team_id
0,20132014,ANA,0.494596,0.482906,24
1,20132014,BOS,0.484655,0.479566,6
2,20132014,BUF,0.518278,0.479450,7
3,20132014,CAR,0.498154,0.485708,12
4,20132014,CBJ,0.498105,0.497083,29


### Special Teams

In [67]:
all_pk[all_pk['season_id']==20232024].head()

,season_id,game_id,team_id,gameDate,gamesPlayed,homeRoad,losses,opponentTeamAbbrev,otLosses,penaltyKillNetPct,penaltyKillPct,pkNetGoals,pkNetGoalsPerGame,pkTimeOnIcePerGame,points,pointsPct,ppGoalsAgainst,ppGoalsAgainstPerGame,shGoalsFor,shGoalsForPerGame,teamFullName,ties,timesShorthanded,timesShorthandedPerGame,wins
24054,20232024,2023020004,9,2023-10-11,1,R,1,CAR,0,1.00,0.83333,0,0.0,663.0,0,0.0,1,1.0,1,1.0,Ottawa Senators,None,6,6.0,0
24055,20232024,2023020004,12,2023-10-11,1,H,0,OTT,0,1.25,1.00000,1,1.0,480.0,2,1.0,0,0.0,1,1.0,Carolina Hurricanes,None,4,4.0,1
24056,20232024,2023020005,10,2023-10-11,1,H,0,MTL,0,0.75,0.75000,-1,-1.0,359.0,2,1.0,1,1.0,0,0.0,Toronto Maple Leafs,None,4,4.0,1
24057,20232024,2023020005,8,2023-10-11,1,R,0,TOR,1,0.50,0.50000,-2,-2.0,331.0,1,0.5,2,2.0,0,0.0,Montréal Canadiens,None,4,4.0,0
24058,20232024,2023020006,16,2023-10-11,1,R,1,BOS,0,1.00,1.00000,0,0.0,360.0,0,0.0,0,0.0,0,0.0,Chicago Blackhawks,None,3,3.0,0


In [ ]:
tx_pp = all_pp[
    all_pp['game_id'].isin(thanksgiving_game_ids)
]



,season_id,game_id,team_id,gameDate,gamesPlayed,homeRoad,losses,opponentTeamAbbrev,otLosses,pointPct,points,powerPlayGoalsFor,powerPlayNetPct,powerPlayPct,ppGoalsPerGame,ppNetGoals,ppNetGoalsPerGame,ppOpportunities,ppOpportunitiesPerGame,ppTimeOnIcePerGame,shGoalsAgainst,shGoalsAgainstPerGame,teamFullName,ties,wins
24054,20232024,2023020004,12,2023-10-11,1,H,0,OTT,0,1.0,2,1,0.000000,0.16666,1.0,0,0.0,6,6.0,663.0,1,1.0,Carolina Hurricanes,None,1
24055,20232024,2023020004,9,2023-10-11,1,R,1,CAR,0,0.0,0,0,-0.250000,0.00000,0.0,-1,-1.0,4,4.0,480.0,1,1.0,Ottawa Senators,None,0
24056,20232024,2023020005,10,2023-10-11,1,H,0,MTL,0,1.0,2,2,0.500000,0.50000,2.0,2,2.0,4,4.0,331.0,0,0.0,Toronto Maple Leafs,None,1
24057,20232024,2023020005,8,2023-10-11,1,R,0,TOR,1,0.5,1,1,0.250000,0.25000,1.0,1,1.0,4,4.0,359.0,0,0.0,Montréal Canadiens,None,0
24058,20232024,2023020006,16,2023-10-11,1,R,1,BOS,0,0.0,0,0,0.000000,0.00000,0.0,0,0.0,3,3.0,360.0,0,0.0,Chicago Blackhawks,None,0
24059,20232024,2023020006,6,2023-10-11,1,H,0,CHI,0,1.0,2,0,0.000000,0.00000,0.0,0,0.0,3,3.0,360.0,0,0.0,Boston Bruins,None,1
24060,20232024,2023020007,20,2023-10-11,1,H,0,WPG,0,1.0,2,1,0.333333,0.33333,1.0,1,1.0,3,3.0,326.0,0,0.0,Calgary Flames,None,1
24061,20232024,2023020007,52,2023-10-11,1,R,1,CGY,0,0.0,0,0,-0.250000,0.00000,0.0,-1,-1.0,4,4.0,480.0,1,1.0,Winnipeg Jets,None,0
24062,20232024,2023020008,26,2023-10-11,1,H,1,COL,0,0.0,0,0,0.000000,0.00000,0.0,0,0.0,5,5.0,600.0,0,0.0,Los Angeles Kings,None,0
24063,20232024,2023020008,21,2023-10-11,1,R,0,LAK,0,1.0,2,1,0.333333,0.33333,1.0,1,1.0,3,3.0,270.0,0,0.0,Colorado Avalanche,None,1


In [69]:

tx_pk = all_pk[
    all_pk['game_id'].isin(thanksgiving_game_ids)
]



In [70]:
tx_pk[tx_pk['season_id']==20232024]

,season_id,game_id,team_id,gameDate,gamesPlayed,homeRoad,losses,opponentTeamAbbrev,otLosses,penaltyKillNetPct,penaltyKillPct,pkNetGoals,pkNetGoalsPerGame,pkTimeOnIcePerGame,points,pointsPct,ppGoalsAgainst,ppGoalsAgainstPerGame,shGoalsFor,shGoalsForPerGame,teamFullName,ties,timesShorthanded,timesShorthandedPerGame,wins
24054,20232024,2023020004,9,2023-10-11,1,R,1,CAR,0,1.000000,0.83333,0,0.0,663.0,0,0.0,1,1.0,1,1.0,Ottawa Senators,None,6,6.0,0
24055,20232024,2023020004,12,2023-10-11,1,H,0,OTT,0,1.250000,1.00000,1,1.0,480.0,2,1.0,0,0.0,1,1.0,Carolina Hurricanes,None,4,4.0,1
24056,20232024,2023020005,10,2023-10-11,1,H,0,MTL,0,0.750000,0.75000,-1,-1.0,359.0,2,1.0,1,1.0,0,0.0,Toronto Maple Leafs,None,4,4.0,1
24057,20232024,2023020005,8,2023-10-11,1,R,0,TOR,1,0.500000,0.50000,-2,-2.0,331.0,1,0.5,2,2.0,0,0.0,Montréal Canadiens,None,4,4.0,0
24058,20232024,2023020006,16,2023-10-11,1,R,1,BOS,0,1.000000,1.00000,0,0.0,360.0,0,0.0,0,0.0,0,0.0,Chicago Blackhawks,None,3,3.0,0
24059,20232024,2023020006,6,2023-10-11,1,H,0,CHI,0,1.000000,1.00000,0,0.0,360.0,2,1.0,0,0.0,0,0.0,Boston Bruins,None,3,3.0,1
24060,20232024,2023020007,52,2023-10-11,1,R,1,CGY,0,0.666666,0.66666,-1,-1.0,326.0,0,0.0,1,1.0,0,0.0,Winnipeg Jets,None,3,3.0,0
24061,20232024,2023020007,20,2023-10-11,1,H,0,WPG,0,1.250000,1.00000,1,1.0,480.0,2,1.0,0,0.0,1,1.0,Calgary Flames,None,4,4.0,1
24062,20232024,2023020008,26,2023-10-11,1,H,1,COL,0,0.666666,0.66666,-1,-1.0,270.0,0,0.0,1,1.0,0,0.0,Los Angeles Kings,None,3,3.0,0
24063,20232024,2023020008,21,2023-10-11,1,R,0,LAK,0,1.000000,1.00000,0,0.0,600.0,2,1.0,0,0.0,0,0.0,Colorado Avalanche,None,5,5.0,1


In [71]:
tx_pk = tx_pk[tx_pk['season_id']!=20252026]
tx_pp = tx_pp[tx_pp['season_id']!=20252026]

In [72]:
tx_pk.columns

Index(['season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses',
       'penaltyKillNetPct', 'penaltyKillPct', 'pkNetGoals',
       'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points', 'pointsPct',
       'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame', 'teamFullName', 'ties', 'timesShorthanded',
       'timesShorthandedPerGame', 'wins'],
      dtype='object')

In [73]:
pp_num_cols = [
    'season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses', 'pointPct',
       'points', 'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
       'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame', 'ppOpportunities',
       'ppOpportunitiesPerGame', 'ppTimeOnIcePerGame', 'shGoalsAgainst',
       'shGoalsAgainstPerGame', 'teamFullName', 'ties', 'wins'
]

pk_num_cols = ['season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses',
       'penaltyKillNetPct', 'penaltyKillPct', 'pkNetGoals',
       'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points', 'pointsPct',
       'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame', 'teamFullName', 'ties', 'timesShorthanded',
       'timesShorthandedPerGame', 'wins']

tx_pp[pp_num_cols] = tx_pp[pp_num_cols].fillna(0)
tx_pk[pk_num_cols] = tx_pk[pk_num_cols].fillna(0)

C:\Users\kjchr\AppData\Local\Temp\ipykernel_7884\4117000422.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tx_pp[pp_num_cols] = tx_pp[pp_num_cols].fillna(0)
C:\Users\kjchr\AppData\Local\Temp\ipykernel_7884\4117000422.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tx_pk[pk_num_cols] = tx_pk[pk_num_cols].fillna(0)


In [74]:
tx_pp_stats = tx_pp.copy()

tx_pp_stats = tx_pp_stats[['season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses', 'pointPct',
       'points', 'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
       'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame', 'ppOpportunities',
       'ppOpportunitiesPerGame', 'ppTimeOnIcePerGame', 'shGoalsAgainst',
       'shGoalsAgainstPerGame', 'teamFullName', 'ties', 'wins']]

tx_pp_stats.head()

,season_id,game_id,team_id,gameDate,gamesPlayed,homeRoad,losses,opponentTeamAbbrev,otLosses,pointPct,points,powerPlayGoalsFor,powerPlayNetPct,powerPlayPct,ppGoalsPerGame,ppNetGoals,ppNetGoalsPerGame,ppOpportunities,ppOpportunitiesPerGame,ppTimeOnIcePerGame,shGoalsAgainst,shGoalsAgainstPerGame,teamFullName,ties,wins
0,20132014,2013020001,10,2013-10-01,1,R,0,MTL,0,1.0,2,1,0.142857,0.14285,1.0,1,1.0,7,7.0,700.0,0,0.0,Toronto Maple Leafs,0,1
1,20132014,2013020001,8,2013-10-01,1,H,1,TOR,0,0.0,0,0,-0.250000,0.00000,0.0,-1,-1.0,4,4.0,405.0,1,1.0,Montréal Canadiens,0,0
2,20132014,2013020002,15,2013-10-01,1,R,1,CHI,0,0.0,0,3,0.500000,0.50000,3.0,3,3.0,6,6.0,388.0,0,0.0,Washington Capitals,0,0
3,20132014,2013020002,16,2013-10-01,1,H,0,WSH,0,1.0,2,1,0.250000,0.25000,1.0,1,1.0,4,4.0,387.0,0,0.0,Chicago Blackhawks,0,1
4,20132014,2013020003,22,2013-10-01,1,H,1,WPG,0,0.0,0,2,0.400000,0.40000,2.0,2,2.0,5,5.0,399.0,0,0.0,Edmonton Oilers,0,0


In [75]:
# ==============================================================
# AGGREGATE PP STATS BY TEAM + SEASON
# ==============================================================

tx_pp_stats = tx_pp_stats[['game_id', 'season_id', 'team_id', 
                     'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
                     'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame',
                     'ppOpportunities', 'ppOpportunitiesPerGame',
                     'ppTimeOnIcePerGame', 'shGoalsAgainst',
                     'shGoalsAgainstPerGame', 'opponentTeamAbbrev']].copy()

# --------------------------------------------------------------
# 1. Aggregate raw totals (correct way)
# --------------------------------------------------------------
tx_agg_pp = (
    tx_pp_stats.groupby(['season_id', 'team_id'], as_index=False)
    .agg(
        games_played=('game_id', 'nunique'),
        powerPlayGoalsFor=('powerPlayGoalsFor', 'sum'),
        shGoalsAgainst=('shGoalsAgainst', 'sum'),
        ppOpportunities=('ppOpportunities', 'sum'),
        ppNetGoals=('ppNetGoals', 'sum'),
        ppTimeOnIce=('ppTimeOnIcePerGame', 'sum'),
    )
)

# --------------------------------------------------------------
# 2. Recompute correct percentages 
# --------------------------------------------------------------

# PP% = PPGF / PPO
tx_agg_pp['powerPlayPct'] = (
    tx_agg_pp['powerPlayGoalsFor'] / tx_agg_pp['ppOpportunities']
).replace([np.inf, -np.inf], np.nan).round(4) * 100

# Net PP% = (PPGF - SHGA) / PPO
tx_agg_pp['powerPlayNetPct'] = (
    (tx_agg_pp['powerPlayGoalsFor'] - tx_agg_pp['shGoalsAgainst'])
    / tx_agg_pp['ppOpportunities']
).replace([np.inf, -np.inf], np.nan).round(4) * 100

# --------------------------------------------------------------
# 3. Recompute per-game metrics
# --------------------------------------------------------------

tx_agg_pp['ppGoalsPerGame'] = (
    tx_agg_pp['powerPlayGoalsFor'] / tx_agg_pp['games_played']
).round(3)

tx_agg_pp['ppOpportunitiesPerGame'] = (
    tx_agg_pp['ppOpportunities'] / tx_agg_pp['games_played']
).round(3)

tx_agg_pp['shGoalsAgainstPerGame'] = (
    tx_agg_pp['shGoalsAgainst'] / tx_agg_pp['games_played']
).round(3)

tx_agg_pp['ppNetGoalsPerGame'] = (
    tx_agg_pp['ppNetGoals'] / tx_agg_pp['games_played']
).round(3)

tx_agg_pp['ppTimeOnIcePerGame'] = (
    tx_agg_pp['ppTimeOnIce'] / tx_agg_pp['games_played']
).round(3)

# Clean final DataFrame
numeric_cols = tx_agg_pp.select_dtypes(include=['float']).columns
tx_agg_pp[numeric_cols] = tx_agg_pp[numeric_cols].round(3)

# Final dataset
tx_agg_pp.head()


,season_id,team_id,games_played,powerPlayGoalsFor,shGoalsAgainst,ppOpportunities,ppNetGoals,ppTimeOnIce,powerPlayPct,powerPlayNetPct,ppGoalsPerGame,ppOpportunitiesPerGame,shGoalsAgainstPerGame,ppNetGoalsPerGame,ppTimeOnIcePerGame
0,20132014,1,60,35,1,177,34,17935.0,19.77,19.21,0.583,2.950,0.017,0.567,298.917
1,20132014,2,61,35,5,209,30,20909.0,16.75,14.35,0.574,3.426,0.082,0.492,342.770
2,20132014,3,60,39,5,195,34,19027.0,20.00,17.44,0.650,3.250,0.083,0.567,317.117
3,20132014,4,60,42,10,219,32,21930.0,19.18,14.61,0.700,3.650,0.167,0.533,365.500
4,20132014,5,59,50,3,194,47,18306.0,25.77,24.23,0.847,3.288,0.051,0.797,310.271


In [76]:
tx_pk_stats = tx_pk.copy()

tx_pk_stats = tx_pk_stats[['season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses',
       'penaltyKillNetPct', 'penaltyKillPct', 'pkNetGoals',
       'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points', 'pointsPct',
       'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame', 'teamFullName', 'ties', 'timesShorthanded',
       'timesShorthandedPerGame', 'wins']]

tx_pk_stats.head()

,season_id,game_id,team_id,gameDate,gamesPlayed,homeRoad,losses,opponentTeamAbbrev,otLosses,penaltyKillNetPct,penaltyKillPct,pkNetGoals,pkNetGoalsPerGame,pkTimeOnIcePerGame,points,pointsPct,ppGoalsAgainst,ppGoalsAgainstPerGame,shGoalsFor,shGoalsForPerGame,teamFullName,ties,timesShorthanded,timesShorthandedPerGame,wins
0,20132014,2013020001,10,2013-10-01,1,R,0,MTL,0,1.250000,1.00000,1,1.0,405.0,2,1.0,0,0.0,1,1.0,Toronto Maple Leafs,0,4,4.0,1
1,20132014,2013020001,8,2013-10-01,1,H,1,TOR,0,0.857142,0.85714,-1,-1.0,700.0,0,0.0,1,1.0,0,0.0,Montréal Canadiens,0,7,7.0,0
2,20132014,2013020002,16,2013-10-01,1,H,0,WSH,0,0.500000,0.50000,-3,-3.0,388.0,2,1.0,3,3.0,0,0.0,Chicago Blackhawks,0,6,6.0,1
3,20132014,2013020002,15,2013-10-01,1,R,1,CHI,0,0.750000,0.75000,-1,-1.0,387.0,0,0.0,1,1.0,0,0.0,Washington Capitals,0,4,4.0,0
4,20132014,2013020003,22,2013-10-01,1,H,1,WPG,0,0.750000,0.75000,-1,-1.0,400.0,0,0.0,1,1.0,0,0.0,Edmonton Oilers,0,4,4.0,0


In [77]:
# ==============================================
# AGGREGATE PK STATS BY TEAM + SEASON
# ==============================================

# Start with your selected PK stats
tx_pk_stats = tx_pk_stats[['season_id', 'game_id', 'team_id', 'gameDate', 'gamesPlayed',
       'homeRoad', 'losses', 'opponentTeamAbbrev', 'otLosses',
       'penaltyKillNetPct', 'penaltyKillPct', 'pkNetGoals',
       'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points', 'pointsPct',
       'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame', 'ties', 'timesShorthanded',
       'timesShorthandedPerGame', 'wins']].copy()

# Group by team + season
tx_agg_pk = (
    tx_pk_stats.groupby(['season_id', 'team_id'], as_index=False)
    .agg(
        games_played=('game_id', 'nunique'),
        ppGoalsAgainst=('ppGoalsAgainst', 'sum'),
        shGoalsFor=('shGoalsFor', 'sum'),
        timesShorthanded=('timesShorthanded', 'sum'),
        pkNetGoals=('pkNetGoals', 'sum'),
        pkTimeOnIce=('pkTimeOnIcePerGame', 'sum'),   # total minutes short-handed
        points=('points', 'sum'),
    )
)

# ----------------------------------------------
# RE-CALCULATE ALL % METRICS PROPERLY
# ----------------------------------------------

# Classic PK% = 1 - PPGA / TSH
tx_agg_pk['penaltyKillPct'] = (
    1 - (tx_agg_pk['ppGoalsAgainst'] / tx_agg_pk['timesShorthanded'])
).round(4) * 100

# Net PK% = (TSH - PPGA + SHGF) / TSH
tx_agg_pk['penaltyKillNetPct'] = (
    (tx_agg_pk['timesShorthanded'] - tx_agg_pk['ppGoalsAgainst'] + tx_agg_pk['shGoalsFor'])
    / tx_agg_pk['timesShorthanded']
).round(4) * 100

# Per-game metrics (recomputed)
tx_agg_pk['ppGoalsAgainstPerGame'] = tx_agg_pk['ppGoalsAgainst'] / tx_agg_pk['games_played']
tx_agg_pk['shGoalsForPerGame'] = tx_agg_pk['shGoalsFor'] / tx_agg_pk['games_played']
tx_agg_pk['timesShorthandedPerGame'] = tx_agg_pk['timesShorthanded'] / tx_agg_pk['games_played']
tx_agg_pk['pkNetGoalsPerGame'] = tx_agg_pk['pkNetGoals'] / tx_agg_pk['games_played']
tx_agg_pk['pkTimeOnIcePerGame'] = tx_agg_pk['pkTimeOnIce'] / tx_agg_pk['games_played']

# Points%
tx_agg_pk['pointsPct'] = tx_agg_pk['points'] / (tx_agg_pk['games_played'] * 2)

# Round final numbers
numeric_cols = tx_agg_pk.select_dtypes(include=['float']).columns
tx_agg_pk[numeric_cols] = tx_agg_pk[numeric_cols].round(3)

# Final dataset
tx_agg_pk.head()


,season_id,team_id,games_played,ppGoalsAgainst,shGoalsFor,timesShorthanded,pkNetGoals,pkTimeOnIce,points,penaltyKillPct,penaltyKillNetPct,ppGoalsAgainstPerGame,shGoalsForPerGame,timesShorthandedPerGame,pkNetGoalsPerGame,pkTimeOnIcePerGame,pointsPct
0,20132014,1,60,24,5,193,-19,20501.0,63,87.56,90.16,0.400,0.083,3.217,-0.317,341.683,0.525
1,20132014,2,61,40,7,183,-33,18027.0,54,78.14,81.97,0.656,0.115,3.000,-0.541,295.525,0.443
2,20132014,3,60,30,3,177,-27,17101.0,69,83.05,84.75,0.500,0.050,2.950,-0.450,285.017,0.575
3,20132014,4,60,38,5,238,-33,24482.0,66,84.03,86.13,0.633,0.083,3.967,-0.550,408.033,0.550
4,20132014,5,59,25,3,181,-22,18102.0,84,86.19,87.85,0.424,0.051,3.068,-0.373,306.814,0.712


In [78]:
tx_special_teams = tx_agg_pp.merge(
    tx_agg_pk,
    on=['season_id', 'team_id'],
    how='inner',
    suffixes=('_pp', '_pk')   # optional — only needed if any column names overlap
)

tx_special_teams.head()

,season_id,team_id,games_played_pp,powerPlayGoalsFor,shGoalsAgainst,ppOpportunities,ppNetGoals,ppTimeOnIce,powerPlayPct,powerPlayNetPct,ppGoalsPerGame,ppOpportunitiesPerGame,shGoalsAgainstPerGame,ppNetGoalsPerGame,ppTimeOnIcePerGame,games_played_pk,ppGoalsAgainst,shGoalsFor,timesShorthanded,pkNetGoals,pkTimeOnIce,points,penaltyKillPct,penaltyKillNetPct,ppGoalsAgainstPerGame,shGoalsForPerGame,timesShorthandedPerGame,pkNetGoalsPerGame,pkTimeOnIcePerGame,pointsPct
0,20132014,1,60,35,1,177,34,17935.0,19.77,19.21,0.583,2.950,0.017,0.567,298.917,60,24,5,193,-19,20501.0,63,87.56,90.16,0.400,0.083,3.217,-0.317,341.683,0.525
1,20132014,2,61,35,5,209,30,20909.0,16.75,14.35,0.574,3.426,0.082,0.492,342.770,61,40,7,183,-33,18027.0,54,78.14,81.97,0.656,0.115,3.000,-0.541,295.525,0.443
2,20132014,3,60,39,5,195,34,19027.0,20.00,17.44,0.650,3.250,0.083,0.567,317.117,60,30,3,177,-27,17101.0,69,83.05,84.75,0.500,0.050,2.950,-0.450,285.017,0.575
3,20132014,4,60,42,10,219,32,21930.0,19.18,14.61,0.700,3.650,0.167,0.533,365.500,60,38,5,238,-33,24482.0,66,84.03,86.13,0.633,0.083,3.967,-0.550,408.033,0.550
4,20132014,5,59,50,3,194,47,18306.0,25.77,24.23,0.847,3.288,0.051,0.797,310.271,59,25,3,181,-22,18102.0,84,86.19,87.85,0.424,0.051,3.068,-0.373,306.814,0.712


In [79]:
tx_special_teams = tx_special_teams.rename(
            columns={
                "powerPlayGoalsFor": "pp_gf",
                "powerPlayNetPct": "pp_net_pct",
                "powerPlayPct": "pp_pct",
                "ppGoalsPerGame": "ppg_g",
                "ppNetGoals": "ppg_net",
                "ppNetGoalsPerGame": "ppg_net_g",
                "ppOpportunities": "pp_opp",
                "ppOpportunitiesPerGame": "pp_opp_g",
                "penaltyKillNetPct": "pk_net_pct",
                "penaltyKillPct": "pk_pct",
                "pkNetGoals": "pkg_net",
                "pkNetGoalsPerGame": "pkg_net_g",
                "ppGoalsAgainst": "ppga",
                "ppGoalsAgainstPerGame": "ppga_g",
                "shGoalsFor": "shgf",
                "shGoalsForPerGame": "shgf_g",
            }
        )

tx_special_teams=tx_special_teams.drop(columns=['points'])
tx_special_teams.head()

,season_id,team_id,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct
0,20132014,1,60,35,1,177,34,17935.0,19.77,19.21,0.583,2.950,0.017,0.567,298.917,60,24,5,193,-19,20501.0,87.56,90.16,0.400,0.083,3.217,-0.317,341.683,0.525
1,20132014,2,61,35,5,209,30,20909.0,16.75,14.35,0.574,3.426,0.082,0.492,342.770,61,40,7,183,-33,18027.0,78.14,81.97,0.656,0.115,3.000,-0.541,295.525,0.443
2,20132014,3,60,39,5,195,34,19027.0,20.00,17.44,0.650,3.250,0.083,0.567,317.117,60,30,3,177,-27,17101.0,83.05,84.75,0.500,0.050,2.950,-0.450,285.017,0.575
3,20132014,4,60,42,10,219,32,21930.0,19.18,14.61,0.700,3.650,0.167,0.533,365.500,60,38,5,238,-33,24482.0,84.03,86.13,0.633,0.083,3.967,-0.550,408.033,0.550
4,20132014,5,59,50,3,194,47,18306.0,25.77,24.23,0.847,3.288,0.051,0.797,310.271,59,25,3,181,-22,18102.0,86.19,87.85,0.424,0.051,3.068,-0.373,306.814,0.712


### Standings at Thanksgiving

In [80]:


# # ================================================================
# # THANKSGIVING DATES BY SEASON
# # ================================================================
# thanksgiving_dates = {
#     # 20132014 : '2014-04-13'
#     # 20132014 : '2013-11-28',
#     # 20142015 : '2014-11-27',
#     # 20152016 : '2015-11-26',
#     # 20162017 : '2016-11-24',
#     # 20172018 : '2017-11-23',
#     # 20182019 : '2018-11-22',
#     20192020 : '2019-11-26',
#     # 20192020 : '2020-03-10',
#     # 20202021 : '2021-02-15',  # COVID modified
#     # 20212022 : '2021-11-25',
#     # 20222023 : '2022-11-24',
#     # 20232024 : '2023-11-23',
#     # 20242025 : '2024-11-28'
# }

# # ================================================================
# # FUNCTION TO FETCH STANDINGS FOR A GIVEN DATE
# # ================================================================
# def get_standings_for_date(standings_date):
#     base_url = "https://api-web.nhle.com/v1/standings/"
#     api_url = f"{base_url}{standings_date}"

#     response = requests.get(api_url, timeout=20)

#     if response.status_code != 200:
#         print(f"❌ Failed for {standings_date}: {response.status_code}")
#         return None

#     try:
#         json_data = response.json()
#     except Exception as e:
#         print(f"❌ JSON Decode Error for {standings_date}: {e}")
#         print(response.text[:200])
#         return None

#     standings = json_data.get("standings", [])
#     df = pd.DataFrame(standings)

#     if df.empty:
#         print(f"⚠️ No standings available for {standings_date}")
#         return None

#     # Extract nested fields
#     df["team_name"] = df["teamName"].apply(lambda x: x["default"])
#     df["tri_code"] = df["teamAbbrev"].apply(lambda x: x["default"])
#     df["season_id"] = df["seasonId"]

#     df["thanksgiving_date"] = standings_date

#     # Keep only necessary fields
#     df = df[
#         [
#             "conferenceName",
#             "conferenceSequence",
#             "team_name",
#             "tri_code",
#             "gamesPlayed",
#             "points",
#             "winPctg",
#             "pointPctg",
#             "regulationPlusOtWins",
#             "goalDifferential",
#             "goalFor",
#             "goalAgainst",
#             "homeGamesPlayed",
#             "homeWins",
#             "roadGamesPlayed",
#             "roadWins",
#             "l10GoalDifferential",
#             "seasonId",
#             "thanksgiving_date",
#             "streakCode",
#             "streakCount",

#         ]
#     ]

#     # Derived stats
#     df["home_win_pctg"] = df["homeWins"] / df["homeGamesPlayed"].replace(0, np.nan)
#     df["road_win_pctg"] = df["roadWins"] / df["roadGamesPlayed"].replace(0, np.nan)
#     df["gf_g"] = df["goalFor"] / df["gamesPlayed"].replace(0, np.nan)
#     df["ga_g"] = df["goalAgainst"] / df["gamesPlayed"].replace(0, np.nan)

#     points_per_game = df["points"] / df["gamesPlayed"].replace(0, np.nan)
#     games_remaining = 82 - df["gamesPlayed"]
#     projected_final_points = df["points"] + (games_remaining * points_per_game)

#     df["projected_points"] = projected_final_points.round()

#     return df

# # ================================================================
# # LOOP THROUGH ALL THANKSGIVING DATES
# # ================================================================
# all_thanksgiving = []

# for season_key, tg_date in thanksgiving_dates.items():
#     print(f"\n📅 Pulling standings for {season_key} — Thanksgiving: {tg_date}")
    
#     df = get_standings_for_date(tg_date)

#     if df is not None and not df.empty:
#         df["season_key"] = season_key
#         all_thanksgiving.append(df)
#     else:
#         print(f"⚠️ No data returned for {tg_date}")

# # ================================================================
# # FINAL COMBINED DATAFRAME
# # ================================================================
# thanksgiving_standings_df = pd.concat(all_thanksgiving, ignore_index=True)



# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\1920_tx_standings.csv"

# # Ensure the folder exists before writing the file
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# thanksgiving_standings_df.to_csv(export_path, index=False)




In [81]:

thanksgiving_standings_df = thanksgiving_standings_df.merge(tx_teams, on = 'tri_code', how="left")
# thanksgiving_standings_df = thanksgiving_standings_df.rename(columns={'season_key':'season_id'})
thanksgiving_standings_df.tail()

,season_id,tri_code,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,thanksgiving_date,team_id
336,20242025,PIT,Pittsburgh Penguins,Eastern,15,24,20,0.333333,0.416667,7,-33,64,97,13,5,11,3,-20,W,1,0.384615,0.272727,2.666667,4.041667,68.0,2024-11-28,5
337,20242025,MTL,Montréal Canadiens,Eastern,16,22,19,0.363636,0.431818,8,-21,62,83,12,5,10,3,-4,W,1,0.416667,0.300000,2.818182,3.772727,71.0,2024-11-28,8
338,20242025,SJS,San Jose Sharks,Western,14,25,19,0.280000,0.380000,7,-23,66,89,12,5,13,2,-3,L,1,0.416667,0.153846,2.640000,3.560000,62.0,2024-11-28,28
339,20242025,CHI,Chicago Blackhawks,Western,15,22,18,0.363636,0.409091,7,-9,56,65,9,4,13,4,-3,W,1,0.444444,0.307692,2.545455,2.954545,67.0,2024-11-28,16
340,20242025,NSH,Nashville Predators,Western,16,23,18,0.304348,0.391304,7,-19,54,73,12,5,11,2,-6,OT,1,0.416667,0.181818,2.347826,3.173913,64.0,2024-11-28,18


## TX Final DF  

In [82]:
tx_final_df = (
    thanksgiving_standings_df[
        ['season_id', 'tri_code', 'team_id', 'team_name', 'conferenceName',
       'conferenceSequence', 'gamesPlayed', 'points', 'winPctg', 'pointPctg',
       'regulationPlusOtWins', 'goalDifferential', 'goalFor', 'goalAgainst',
       'homeGamesPlayed', 'homeWins', 'roadGamesPlayed', 'roadWins',
       'l10GoalDifferential', 'streakCode', 'streakCount', 'home_win_pctg',
       'road_win_pctg', 'gf_g', 'ga_g', 'projected_points',
       'thanksgiving_date']
    ]
    .merge(tx_strength, on=['team_id','tri_code','season_id'], how='left')
    .merge(tx_special_teams, on=['team_id','season_id'], how="left")
    .merge(tx_rush_stats, on=['team_id','tri_code','season_id'], how='left')

    .merge(team_goalie_stats_tx, on=['team_id','tri_code','season_id'], how='left')
    .merge(tx_shooting_stats,on=['team_id','season_id'], how='left')
    .merge(tx_final_scoring_stats, on=['team_id','season_id'], how='left')
)
today = date.today()
tx_final_df['run_date'] = today
tx_final_df.head()

,season_id,tri_code,team_id,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,thanksgiving_date,SOS,Remaining_SOS,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct,rush_shots_for,rush_shots_against,rush_shot_differential,team_savePct,team_GAA,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo,SCF,SCSF,SCGF,SCA,run_date
0,20132014,CHI,16,Chicago Blackhawks,Western,1,26,40,0.692308,0.769231,15,22,95,73,13,9,13,9,9,W,4,0.692308,0.692308,3.653846,2.807692,126.0,2013-11-28,0.499729,0.523092,61,42,5,193,37,18737.0,21.76,19.17,0.689,3.164,0.082,0.607,307.164,61,37,6,183,-31,17625.0,79.78,83.06,0.607,0.098,3.000,-0.508,288.934,0.689,392,319,73,0.907404,2.501442,7.82,2799,2331,7.59,92.41,100.23,1761,1145,175,3280.0,2025-12-04
1,20132014,STL,19,St. Louis Blues,Western,2,24,39,0.750000,0.812500,15,35,86,51,14,11,10,7,18,W,5,0.785714,0.700000,3.583333,2.125000,133.0,2013-11-28,0.496685,0.502106,59,45,5,213,40,20019.0,21.13,18.78,0.763,3.610,0.085,0.678,339.305,59,30,2,208,-28,21033.0,85.58,86.54,0.508,0.034,3.525,-0.475,356.492,0.712,300,265,35,0.917035,2.171772,8.25,2497,2242,6.51,93.49,101.74,1613,967,165,3079.0,2025-12-04
2,20132014,SJS,28,San Jose Sharks,Western,3,24,37,0.666667,0.770833,14,28,82,54,11,8,13,8,2,W,3,0.727273,0.615385,3.416667,2.250000,126.0,2013-11-28,0.497561,0.494459,61,38,2,222,36,21768.0,17.12,16.22,0.623,3.639,0.033,0.590,356.852,61,26,5,160,-21,15683.0,83.75,86.88,0.426,0.082,2.623,-0.344,257.098,0.672,404,332,72,0.916663,2.259428,6.40,3046,2419,6.49,93.51,99.91,1906,1227,159,3477.0,2025-12-04
3,20132014,ANA,24,Anaheim Ducks,Western,4,27,37,0.629630,0.685185,16,12,83,71,10,9,17,8,-3,L,1,0.900000,0.470588,3.074074,2.629630,112.0,2013-11-28,0.494596,0.482906,61,37,5,207,32,20967.0,17.87,15.46,0.607,3.393,0.082,0.525,343.721,61,36,6,204,-30,20121.0,82.35,85.29,0.590,0.098,3.344,-0.492,329.852,0.730,332,317,15,0.918173,2.274396,7.68,2629,2558,5.94,94.06,101.74,1845,1124,167,3481.0,2025-12-04
4,20132014,LAK,26,Los Angeles Kings,Western,5,26,36,0.615385,0.692308,12,13,69,56,13,8,13,8,8,OT,1,0.615385,0.615385,2.653846,2.153846,114.0,2013-11-28,0.506371,0.486135,61,30,4,216,26,22165.0,13.89,12.04,0.492,3.541,0.066,0.426,363.361,61,41,5,235,-36,23487.0,82.55,84.68,0.672,0.082,3.852,-0.590,385.033,0.590,339,359,-20,0.923404,2.026257,5.45,2789,2376,5.77,94.23,99.68,1617,1017,119,3155.0,2025-12-04


In [83]:
tx_final_df[tx_final_df['season_id']==20232024]

,season_id,tri_code,team_id,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,thanksgiving_date,SOS,Remaining_SOS,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct,rush_shots_for,rush_shots_against,rush_shot_differential,team_savePct,team_GAA,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo,SCF,SCSF,SCGF,SCA,run_date
277,20232024,BOS,6,Boston Bruins,Eastern,1,18,31,0.777778,0.861111,13,25,63,38,8,7,10,7,11,W,1,0.875000,0.700000,3.500000,2.111111,141.0,2023-11-23,0.462254,0.507301,18,13,1,60,12,5547.0,21.67,20.00,0.722,3.333,0.056,0.667,308.167,18,6,0,67,-6,6756.0,91.04,91.04,0.333,0.000,3.722,-0.333,375.333,0.861,774,759,15,0.932620,2.090175,8.52,2699,2523,5.23,94.77,103.29,2503,1336,195,NaN,2025-12-04
278,20232024,VGK,54,Vegas Golden Knights,Western,1,20,30,0.700000,0.750000,11,21,69,48,10,8,10,6,5,W,1,0.800000,0.600000,3.450000,2.400000,123.0,2023-11-23,0.490990,0.483441,19,16,1,65,15,6257.0,24.62,23.08,0.842,3.421,0.053,0.789,329.316,19,9,4,58,-5,6240.0,84.48,91.38,0.474,0.211,3.053,-0.263,328.421,0.737,818,790,28,0.926870,2.192845,7.23,2711,2590,6.80,93.20,100.43,2457,1274,165,NaN,2025-12-04
279,20232024,NYR,3,New York Rangers,Eastern,2,17,27,0.764706,0.794118,12,16,56,40,6,5,11,8,8,W,1,0.833333,0.727273,3.294118,2.352941,130.0,2023-11-23,0.436746,0.508965,17,17,3,55,14,5018.0,30.91,25.45,1.000,3.235,0.176,0.824,295.176,17,8,1,54,-7,5456.0,85.19,87.04,0.471,0.059,3.176,-0.412,320.941,0.794,830,827,3,0.923391,2.155337,7.50,2772,2460,7.03,92.97,100.47,2419,1175,159,NaN,2025-12-04
280,20232024,VAN,23,Vancouver Canucks,Western,2,20,27,0.650000,0.675000,13,29,80,51,9,7,11,6,5,L,1,0.777778,0.545455,4.000000,2.550000,111.0,2023-11-23,0.450125,0.497466,20,22,0,74,22,7008.0,29.73,29.73,1.100,3.700,0.000,1.100,350.400,20,15,3,64,-12,5796.0,76.56,81.25,0.750,0.150,3.200,-0.600,289.800,0.675,676,692,-16,0.921049,2.457849,8.11,2551,2634,9.23,90.77,98.88,2311,1188,178,NaN,2025-12-04
281,20232024,DAL,25,Dallas Stars,Western,3,18,26,0.666667,0.722222,11,12,62,50,9,5,9,7,7,OT,1,0.555556,0.777778,3.444444,2.777778,118.0,2023-11-23,0.520736,0.482583,18,12,3,59,9,5283.0,20.34,15.25,0.667,3.278,0.167,0.500,293.500,18,8,6,63,-2,6280.0,87.30,96.83,0.444,0.333,3.500,-0.111,348.889,0.722,827,830,-3,0.918540,2.607889,7.46,2723,2652,6.15,93.85,101.31,2530,1286,165,NaN,2025-12-04
282,20232024,LAK,26,Los Angeles Kings,Western,4,17,25,0.647059,0.735294,11,22,67,45,9,3,8,8,16,W,3,0.333333,1.000000,3.941176,2.647059,121.0,2023-11-23,0.555169,0.464334,17,13,3,67,10,6916.0,19.40,14.93,0.765,3.941,0.176,0.588,406.824,17,6,4,61,-2,6616.0,90.16,96.72,0.353,0.235,3.588,-0.118,389.176,0.735,787,792,-5,0.911761,2.450684,7.74,2816,2559,8.40,91.60,99.34,2457,1252,180,NaN,2025-12-04
283,20232024,FLA,13,Florida Panthers,Eastern,3,19,25,0.631579,0.657895,12,6,58,52,9,7,10,5,5,L,1,0.777778,0.500000,3.052632,2.736842,108.0,2023-11-23,0.460517,0.519519,19,10,2,62,8,6208.0,16.13,12.90,0.526,3.263,0.105,0.421,326.737,19,14,1,60,-13,5614.0,76.67,78.33,0.737,0.053,3.158,-0.684,295.474,0.658,842,810,32,0.907016,2.605148,6.90,3143,2777,7.85,92.15,99.05,2797,1491,185,NaN,2025-12-04
284,20232024,COL,21,Colorado Avalanche,Western,5,18,24,0.666667,0.666667,11,15,70,55,8,6,10,6,7,W,1,0.750000,0.600000,3.888889,3.055556,109.0,2023-11-23,0.486195,0.487063,18,14,4,71,10,6635.0,19.72,14.08,0.778,3.944,0.222,0.556,368.611,18,9,5,71,-4,6949.0,87.32,94.37,0.500,0.278,3.944,-0.222,386.056,0.667,756,751,5,0.892211,2.996483,7.12,2614,2543,6.29,93.71,100.83,2316,1185,151,NaN,2025-12-04
285,20232024,W

In [84]:

# Define export path
export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Thanksgiving Prep\tx_final_df.csv"

# Ensure the folder exists before writing the file
os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # Export the data to CSV
tx_final_df.to_csv(export_path, index=False)


## Current Data

### Current Game Summary

In [85]:
# # ============================================================
# # CONFIGURE DATE RANGE
# # ============================================================

# today = pd.to_datetime(date.today())  # today's date for end boundary

# start_date = datetime.strptime("2025-10-07", "%Y-%m-%d")
# end_date   = today   # Keep as datetime

# base_url = "https://api-web.nhle.com/v1/schedule/"

# all_games = []

# # ============================================================
# # LOOP DAY-BY-DAY THROUGH SCHEDULE
# # ============================================================

# current_date = start_date   # <-- separate loop iterator

# while current_date <= end_date:
#     formatted_date = current_date.strftime("%Y-%m-%d")
#     url = f"{base_url}{formatted_date}"

#     response = requests.get(url)

#     if response.status_code != 200:
#         print(f"❌ Failed for {formatted_date}: {response.status_code}")
#         current_date += timedelta(days=1)
#         continue

#     data = response.json()

#     for block in data.get("gameWeek", []):
#         games = block.get("games", [])
#         for g in games:
#             all_games.append(g)

#     current_date += timedelta(days=1)

# print(f"\n📌 Total games found: {len(all_games)}")

# # ============================================================
# # NORMALIZE ALL GAMES INTO ONE DATAFRAME
# # ============================================================

# current_game_summary = pd.json_normalize(all_games)

# # ============================================================
# # FILTER COLUMNS
# # ============================================================

# current_game_summary = current_game_summary[
#     [
#         "id",
#         "season",
#         "startTimeUTC",
#         "gameType",
#         "awayTeam.id",
#         "awayTeam.abbrev",
#         "awayTeam.logo",
#         "homeTeam.id",
#         "homeTeam.abbrev",
#         "homeTeam.logo",
#         "homeTeam.placeName.default",
#         "awayTeam.placeName.default",
#         "awayTeam.score",
#         "homeTeam.score",
#         "winningGoalScorer.playerId",
#         "winningGoalie.playerId",
#         "gameState",
#     ]
# ]

# # ============================================================
# # CLEAN TYPES & CREATE PBP LINK
# # ============================================================

# current_game_summary = current_game_summary.convert_dtypes()
# current_game_summary["id"] = current_game_summary["id"].astype("string")

# current_game_summary = current_game_summary.dropna(subset=["id"])
# current_game_summary = current_game_summary.rename(columns={"id": "game_id"})

# current_game_summary["link"] = (
#     "https://api-web.nhle.com/v1/gamecenter/" + current_game_summary["game_id"] + "/play-by-play"
# )

# current_game_summary = current_game_summary.sort_values("game_id").reset_index(drop=True)

# # ============================================================
# # TIMEZONE: UTC → EASTERN
# # ============================================================

# current_game_summary["startTimeUTC"] = pd.to_datetime(current_game_summary["startTimeUTC"])
# eastern = pytz.timezone("America/New_York")

# current_game_summary["game_date_time"] = current_game_summary["startTimeUTC"].dt.tz_convert(eastern)
# current_game_summary["game_date"] = current_game_summary["game_date_time"].dt.strftime("%Y-%m-%d")
# current_game_summary["start_time"] = (
#     current_game_summary["game_date_time"].dt.strftime("%I:%M %p")
#     .str.lstrip("0")
#     .str.lower()
# )

# current_game_summary.drop("startTimeUTC", axis=1, inplace=True)

# current_game_summary.head()

# # Export
# if current_game_summary is not None:
#     export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Thanksgiving Prep\current_game_summary.csv"
#     os.makedirs(os.path.dirname(export_path), exist_ok=True)
#     current_game_summary.to_csv(export_path, index=False)
#     print("✅ Final dataset exported successfully!")
# else:
#     print("❌ No data returned.")

### Current Standings

In [86]:
# # today's standings
# base_url = "https://api-web.nhle.com/v1/standings/"
# # current_date = pd.to_datetime(date.today())
# # formatted_date = current_date.strftime("%Y-%m-%d")
# # api_url = f"{base_url}{formatted_date}"

# api_url = f"{base_url}{'2025-11-27'}"
# response = requests.get(api_url)

# # Check if the request was successful (status code 200)
# if response.status_code != 200:
#     print(f"Request failed with status code {response.status_code}")
#     print(response.text[:200])  # optional debug peek
#     raise SystemExit  # or return / handle however you want

# # Safely parse JSON
# try:
#     json_data = response.json()
# except json.JSONDecodeError:
#     print("Could not decode JSON from standings endpoint:")
#     print(response.text[:200])
#     raise

# standings = json_data["standings"]
# standings_df = pd.DataFrame(standings)

# # Extract team names
# standings_df["team_name"] = standings_df["teamName"].apply(lambda x: x["default"])
# standings_df["tri_code"] = standings_df["teamAbbrev"].apply(lambda x: x["default"])
# standings_df["season_id"] = standings_df["seasonId"]
# standings_df["num_teams"] = 32

# standings_df = standings_df[
#     [

#             "conferenceName",
#             "conferenceSequence",
#             "team_name",
#             "tri_code",
#             "gamesPlayed",
#             "points",
#             "winPctg",
#             "pointPctg",
#             "regulationPlusOtWins",
#             "goalDifferential",
#             "goalFor",
#             "goalAgainst",
#             "homeGamesPlayed",
#             "homeWins",
#             "roadGamesPlayed",
#             "roadWins",
#             "l10GoalDifferential",
#             "seasonId",
#             "streakCode",
#             "streakCount",

#         ]
#     ]

# # Avoid divide-by-zero just in case
# standings_df["home_win_pctg"] = standings_df["homeWins"] / standings_df["homeGamesPlayed"].replace(0, np.nan)
# standings_df["road_win_pctg"] = standings_df["roadWins"] / standings_df["roadGamesPlayed"].replace(0, np.nan)

# points_per_game = standings_df["points"] / standings_df["gamesPlayed"].replace(0, np.nan)
# standings_df["gf_g"] = standings_df["goalFor"] / standings_df["gamesPlayed"].replace(0, np.nan)
# standings_df["ga_g"] = standings_df["goalAgainst"] / standings_df["gamesPlayed"].replace(0, np.nan)

# games_remaining = 82 - standings_df["gamesPlayed"]
# projected_final_points = standings_df["points"] + (games_remaining * points_per_game)
# standings_df["projected_points"] = round(projected_final_points)

# current_standings = standings_df
# current_standings.head()

# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Base Data\Standings\current_thanksgiving_standings.csv"

# # Ensure the folder exists before writing the file
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# current_standings.to_csv(export_path, index=False)

# # Export


In [87]:
# current_goalies = goalies_individual_games[goalies_individual_games['season_id']==20252026]
# current_goalies.head()

### Goalies

In [88]:
current_goalies = goalies_individual_games[
    goalies_individual_games['game_id'].isin(game_ids)
]

current_goalies.head()

,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
31394,20252026,2025020163,1,1,Jonathan Quick,8471734,NYR,3600,1,0,0,0.0,23,1,1.0,23,0
31395,20252026,2025020212,1,1,Darcy Kuemper,8475311,LAK,3575,1,0,0,0.0,23,1,1.0,23,0
31396,20252026,2025020130,1,1,Sergei Bobrovsky,8475683,FLA,3588,1,0,0,0.0,17,1,1.0,17,0
31397,20252026,2025020323,1,1,Sergei Bobrovsky,8475683,FLA,3583,1,0,0,0.0,32,1,1.0,32,0
31398,20252026,2025020350,1,1,Scott Wedgewood,8475809,COL,3600,1,0,0,0.0,22,1,1.0,22,0


In [89]:
goalies_individual = current_goalies.groupby(['goalieFullName','teamAbbrev', 'season_id']).agg(
    i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * current_goalies.loc[x.index, 'shotsAgainst']).sum() / current_goalies.loc[x.index, 'shotsAgainst'].sum()),
    i_GAA=pd.NamedAgg(column='goalsAgainstAverage', aggfunc=lambda x: (x * current_goalies.loc[x.index, 'timeOnIce']).sum() / current_goalies.loc[x.index, 'timeOnIce'].sum()),
    i_gamesPlayed=('gamesPlayed', 'sum'),
    i_gamesStarted=('gamesStarted', 'sum'),
    i_wins=('wins', 'sum'),
    i_losses = ('losses', 'sum'),
    i_otlosses = ('otLosses', 'sum'),
    i_shutouts=('shutouts', 'sum'),
    i_saves=('saves', 'sum'),
    i_shotsAgainst=('shotsAgainst', 'sum'),
    i_timeOnIce=("timeOnIce",'sum'),
).sort_values("teamAbbrev").reset_index()
goalies_individual.head()

,goalieFullName,teamAbbrev,season_id,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst,i_timeOnIce
0,Lukas Dostal,ANA,20252026,0.904379,2.813596,17,17,11,5,1,0,454,502,61416
1,Petr Mrazek,ANA,20252026,0.881078,3.689035,6,6,3,3,0,0,163,185,21469
2,Jeremy Swayman,BOS,20252026,0.914848,2.676437,16,16,10,6,0,0,462,505,57838
3,Joonas Korpisalo,BOS,20252026,0.877948,3.430464,9,9,4,5,0,0,223,254,32532
4,Ukko-Pekka Luukkonen,BUF,20252026,0.890801,2.726836,7,6,3,3,1,0,155,174,25084


In [90]:
last_10 = current_goalies.sort_values(['goalieFullName','game_id'], ascending=[True, False])
last_10 = last_10.groupby("goalieFullName").head(10)

last_10.head()


,season_id,game_id,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst
31771,20252026,2025020097,1,1,Adin Hill,8478499,VGK,594,0,0,0,0.00000,4,0,1.00000,4,0
31479,20252026,2025020087,1,1,Adin Hill,8478499,VGK,3598,1,0,0,1.00055,26,0,0.96296,27,1
32106,20252026,2025020053,1,1,Adin Hill,8478499,VGK,1200,0,0,0,6.00000,8,0,0.80000,10,2
31857,20252026,2025020037,1,1,Adin Hill,8478499,VGK,3896,0,0,1,1.84804,20,0,0.90909,22,2
32094,20252026,2025020007,1,1,Adin Hill,8478499,VGK,3893,0,0,1,4.62368,21,0,0.80769,26,5


In [91]:
last_10 = last_10.groupby(['goalieFullName','teamAbbrev', 'season_id']).agg(
    i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * current_goalies.loc[x.index, 'shotsAgainst']).sum() / current_goalies.loc[x.index, 'shotsAgainst'].sum()),
    i_GAA=pd.NamedAgg(column='goalsAgainstAverage', aggfunc=lambda x: (x * current_goalies.loc[x.index, 'timeOnIce']).sum() / current_goalies.loc[x.index, 'timeOnIce'].sum()),
    i_gamesPlayed=('gamesPlayed', 'sum'),
    i_gamesStarted=('gamesStarted', 'sum'),
    i_wins=('wins', 'sum'),
    i_losses = ('losses', 'sum'),
    i_otlosses = ('otLosses', 'sum'),
    i_shutouts=('shutouts', 'sum'),
    i_saves=('saves', 'sum'),
    i_shotsAgainst=('shotsAgainst', 'sum'),
).sort_values("teamAbbrev").reset_index()
last_10.head()

,goalieFullName,teamAbbrev,season_id,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst
0,Lukas Dostal,ANA,20252026,0.903444,2.793246,10,10,8,2,0,0,262,290
1,Petr Mrazek,ANA,20252026,0.881078,3.689035,6,6,3,3,0,0,163,185
2,Jeremy Swayman,BOS,20252026,0.917979,2.572343,10,10,7,3,0,0,291,317
3,Joonas Korpisalo,BOS,20252026,0.877948,3.430464,9,9,4,5,0,0,223,254
4,Ukko-Pekka Luukkonen,BUF,20252026,0.890801,2.726836,7,6,3,3,1,0,155,174


In [92]:

goalie_games_stats = current_goalies.groupby(['teamAbbrev','season_id']).agg(
    team_gamesPlayed=('gamesPlayed', 'sum'),
    team_gamesStarted=('gamesStarted', 'sum'),
    team_wins=('wins', 'sum'),
    team_losses=('losses', 'sum'),
    team_otlosses=('otLosses', 'sum'),
    team_shutouts=('shutouts', 'sum'),
    team_saves=('saves', 'sum'),
    team_shotsAgainst=('shotsAgainst', 'sum'),
    team_timeOnIce=("timeOnIce", 'sum'),
).reset_index()

goalie_games_stats.head()

,teamAbbrev,season_id,team_gamesPlayed,team_gamesStarted,team_wins,team_losses,team_otlosses,team_shutouts,team_saves,team_shotsAgainst,team_timeOnIce
0,ANA,20252026,23,23,14,8,1,0,617,687,82885
1,BOS,20252026,25,25,14,11,0,0,685,759,90370
2,BUF,20252026,24,23,9,10,4,1,632,702,82836
3,CAR,20252026,24,23,14,7,2,1,515,578,83250
4,CBJ,20252026,24,24,11,9,4,0,669,743,87713


In [93]:
team_goalie_stats = goalies_individual.groupby(['teamAbbrev', 'season_id']).agg(
    team_savePct=pd.NamedAgg(
        column='i_savePct',
        aggfunc=lambda x: (
            (x * goalies_individual.loc[x.index, 'i_shotsAgainst']).sum()
            / goalies_individual.loc[x.index, 'i_shotsAgainst'].sum()
        )
    ),
    team_GAA=pd.NamedAgg(
        column='i_GAA',
        aggfunc=lambda x: (
            (x * goalies_individual.loc[x.index, 'i_timeOnIce']).sum()
            / goalies_individual.loc[x.index, 'i_timeOnIce'].sum()
        )
    ),
    team_wins=('i_wins', 'sum'),
    team_losses=('i_losses', 'sum'),
    team_shutouts=('i_shutouts', 'sum'),
    team_saves=('i_saves', 'sum'),
    team_shotsAgainst=('i_shotsAgainst', 'sum'),
).reset_index()


team_goalie_stats.head()

,teamAbbrev,season_id,team_savePct,team_GAA,team_wins,team_losses,team_shutouts,team_saves,team_shotsAgainst
0,ANA,20252026,0.898104,3.040354,14,8,0,617,687
1,BOS,20252026,0.902500,2.947877,14,11,0,685,759
2,BUF,20252026,0.900280,3.042152,9,10,1,632,702
3,CAR,20252026,0.890999,2.724320,14,7,1,515,578
4,CBJ,20252026,0.900399,3.037175,11,9,0,669,743


In [94]:
team_goalie_stats = team_goalie_stats.rename(columns={'teamAbbrev':'tri_code'})
team_goalie_stats = team_goalie_stats[['tri_code', 'team_savePct', 'team_GAA', 'season_id']]
team_goalie_stats.head()

,tri_code,team_savePct,team_GAA,season_id
0,ANA,0.898104,3.040354,20252026
1,BOS,0.902500,2.947877,20252026
2,BUF,0.900280,3.042152,20252026
3,CAR,0.890999,2.724320,20252026
4,CBJ,0.900399,3.037175,20252026


In [95]:
team_goalie_stats=team_goalie_stats.merge(teams, on='tri_code', how='left')
team_goalie_stats.head()


,tri_code,team_savePct,team_GAA,season_id,team_id
0,ANA,0.898104,3.040354,20252026,24
1,BOS,0.902500,2.947877,20252026,6
2,BUF,0.900280,3.042152,20252026,7
3,CAR,0.890999,2.724320,20252026,12
4,CBJ,0.900399,3.037175,20252026,29


### Rush Chances

In [96]:
current_game_plays.head()


,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
0,20252026,2025020001,52,00:00:00,20:00:00,520,period-start,8,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,0,13,16,20252026,2025020001
1,20252026,2025020001,51,00:00:00,20:00:00,502,faceoff,11,1,16.0,0.0,0.0,N,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,0,13,16,20252026,2025020001
2,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001
3,20252026,2025020001,8,00:34:00,19:26:00,516,stoppage,13,1,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,puck-in-crowd,NaN,NaN,5 on 5,in net,34,13,16,20252026,2025020001
4,20252026,2025020001,55,00:34:00,19:26:00,502,faceoff,16,1,16.0,-69.0,22.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,5 on 5,in net,34,13,16,20252026,2025020001


In [97]:
# --------------------------------------
# 1. Define event groups
# --------------------------------------
shot_events = {
    "shot-on-goal", 
    "missed-shot", 
    "goal", 
    "blocked-shot", 
    "failed-shot-attempt"
}

transition_events = {
    "giveaway",
    "takeaway",
    "hit"
}

stoppage_events = {
    "faceoff",
    "stoppage",
    "period-end",
    "game-end",
    "penalty",
    "delayed-penalty",
}

valid_prev_events = shot_events | transition_events | stoppage_events

# --------------------------------------
# 2. Filter to only events we care about
# --------------------------------------
mask = current_game_plays["typeDescKey"].isin(valid_prev_events)
valid_plays = current_game_plays.loc[mask].copy()

# Sort by game + time
valid_plays = valid_plays.sort_values(["game_id", "sortOrder"])

# --------------------------------------
# 3. Build "previous event" context
# --------------------------------------
grouped = valid_plays.groupby("game_id", sort=False)

valid_plays["prev_event_time"] = grouped["game_in_seconds"].shift(1)
valid_plays["prev_event_type"] = grouped["typeDescKey"].shift(1)
valid_plays["prev_zone"]       = grouped["zone_code"].shift(1)
valid_plays["prev_team_id"]    = grouped["team_id"].shift(1)

valid_plays["prev_event_was_stoppage"] = valid_plays["prev_event_type"].isin(stoppage_events)
valid_plays["time_diff"] = valid_plays["game_in_seconds"] - valid_plays["prev_event_time"]
# --------------------------------------
# 4. Rush shot condition
# --------------------------------------
is_shot = valid_plays["typeDescKey"].isin(shot_events)

# Time window: allow 0–5 seconds
quick = valid_plays["time_diff"].between(0, 5, inclusive="both")

# Possession change: previous team != shooting team
possession_change = (
    valid_plays["prev_team_id"].notna() &
    (valid_plays["prev_team_id"] != valid_plays["team_id"])
)

# Same team, but coming from DZ or NZ
same_team_dz_nz = (
    (valid_plays["prev_team_id"] == valid_plays["team_id"]) &
    (valid_plays["prev_zone"].isin(["D", "N"]))
)

no_stoppage = ~valid_plays["prev_event_was_stoppage"]

rush_mask = (
    is_shot &
    quick &
    no_stoppage &
    (possession_change | same_team_dz_nz)
)

rush_df = valid_plays.loc[rush_mask].copy()


rush_df["rush_shot"] = 1


In [98]:
# --------------------------------------
# 5. Rush shots FOR (team taking the rush shot)
# --------------------------------------
rush_shots_for = (
    rush_df
    .groupby(["season_id", "team_id"])
    .size()
    .reset_index(name="rush_shots_for")
)

# --------------------------------------
# 6. Rush shots AGAINST (team whose previous event enabled the rush)
# --------------------------------------
rush_shots_against = (
    rush_df
    .groupby(["season_id", "prev_team_id"])
    .size()
    .reset_index(name="rush_shots_against")
    .rename(columns={"prev_team_id": "team_id"})
)

# --------------------------------------
# 7. Merge FOR + AGAINST
# --------------------------------------
rush_stats = rush_shots_for.merge(
    rush_shots_against,
    on=["season_id", "team_id"],
    how="outer"
)

rush_stats = rush_stats.fillna(0)

rush_stats["rush_shots_for"] = rush_stats["rush_shots_for"].astype(int)
rush_stats["rush_shots_against"] = rush_stats["rush_shots_against"].astype(int)

rush_stats["rush_shot_differential"] = (
    rush_stats["rush_shots_for"] - rush_stats["rush_shots_against"]
)

rush_stats = (
    rush_stats
    .sort_values(["season_id", "rush_shot_differential"], ascending=[True, False])
    .reset_index(drop=True)
)

rush_stats.head()


,season_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential
0,20252026,12.0,2964,2913,51
1,20252026,21.0,2630,2590,40
2,20252026,13.0,2238,2210,28
3,20252026,18.0,2459,2438,21
4,20252026,29.0,2665,2645,20


In [99]:
rush_stats = rush_stats.merge(teams, on = "team_id", how="left")
rush_df_abbrev = rush_df[['game_id', 'eventId', 'game_in_seconds', 'prev_event_time', 'prev_zone', 'prev_event_type', 'prev_team_id', 'prev_event_was_stoppage', 'time_diff', 'rush_shot']]


In [100]:
rush_df_abbrev.head()

,game_id,eventId,game_in_seconds,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
7,2025020001,67,73,70.0,N,giveaway,16.0,False,3.0,1
31,2025020001,163,214,211.0,D,blocked-shot,13.0,False,3.0,1
40,2025020001,182,292,288.0,D,hit,16.0,False,4.0,1
69,2025020001,106,629,625.0,O,hit,16.0,False,4.0,1
71,2025020001,263,638,634.0,D,hit,16.0,False,4.0,1


### All Shot Attempts

In [101]:
all_shots_made = current_game_plays.copy()


In [102]:
all_shots_made = all_shots_made[all_shots_made["typeCode"].isin([505, 506, 507, 508, 537])]
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1
2,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001
7,20252026,2025020001,67,01:13:00,18:47:00,508,blocked-shot,25,1,13.0,-64.0,-16.0,D,NaN,NaN,None,NaN,NaN,NaN,NaN,blocked,8476891.0,NaN,5 on 5,in net,73,13,16,20252026,2025020001
10,20252026,2025020001,71,01:16:00,18:44:00,506,shot-on-goal,29,1,13.0,-58.0,-22.0,O,NaN,NaN,wrist,8481519.0,0.0,1.0,NaN,None,NaN,NaN,5 on 5,in net,76,13,16,20252026,2025020001
15,20252026,2025020001,102,01:35:00,18:25:00,506,shot-on-goal,35,1,13.0,-33.0,-19.0,O,NaN,NaN,slap,8481519.0,0.0,2.0,NaN,None,NaN,NaN,5 on 5,in net,95,13,16,20252026,2025020001
18,20252026,2025020001,80,01:44:00,18:16:00,507,missed-shot,38,1,16.0,55.0,-29.0,O,NaN,NaN,snap,8475683.0,NaN,NaN,NaN,hit-crossbar,NaN,NaN,5 on 5,in net,104,13,16,20252026,2025020001


In [103]:
all_shots_made["goalie"] = all_shots_made["goalie"].fillna(0)

In [104]:
#add in the rush shot attempts
all_shots_made = all_shots_made.merge(rush_df_abbrev, on=['game_id', 'eventId','game_in_seconds'], how = 'left')
all_shots_made['rush_shot']= all_shots_made['rush_shot'].fillna(0)
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot
0,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,20252026,2025020001,67,01:13:00,18:47:00,508,blocked-shot,25,1,13.0,-64.0,-16.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8476891.0,NaN,5 on 5,in net,73,13,16,20252026,2025020001,70.0,N,giveaway,16.0,False,3.0,1.0
2,20252026,2025020001,71,01:16:00,18:44:00,506,shot-on-goal,29,1,13.0,-58.0,-22.0,O,NaN,NaN,wrist,8481519.0,0.0,1.0,NaN,None,NaN,NaN,5 on 5,in net,76,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,20252026,2025020001,102,01:35:00,18:25:00,506,shot-on-goal,35,1,13.0,-33.0,-19.0,O,NaN,NaN,slap,8481519.0,0.0,2.0,NaN,None,NaN,NaN,5 on 5,in net,95,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,20252026,2025020001,80,01:44:00,18:16:00,507,missed-shot,38,1,16.0,55.0,-29.0,O,NaN,NaN,snap,8475683.0,NaN,NaN,NaN,hit-crossbar,NaN,NaN,5 on 5,in net,104,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [105]:
# Identify Corsi shots - those only in the O Zone
all_shots_made['corsi'] = np.where(all_shots_made['zone_code']== "O", 1, 0)
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi
0,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
1,20252026,2025020001,67,01:13:00,18:47:00,508,blocked-shot,25,1,13.0,-64.0,-16.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8476891.0,NaN,5 on 5,in net,73,13,16,20252026,2025020001,70.0,N,giveaway,16.0,False,3.0,1.0,0
2,20252026,2025020001,71,01:16:00,18:44:00,506,shot-on-goal,29,1,13.0,-58.0,-22.0,O,NaN,NaN,wrist,8481519.0,0.0,1.0,NaN,None,NaN,NaN,5 on 5,in net,76,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1
3,20252026,2025020001,102,01:35:00,18:25:00,506,shot-on-goal,35,1,13.0,-33.0,-19.0,O,NaN,NaN,slap,8481519.0,0.0,2.0,NaN,None,NaN,NaN,5 on 5,in net,95,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1
4,20252026,2025020001,80,01:44:00,18:16:00,507,missed-shot,38,1,16.0,55.0,-29.0,O,NaN,NaN,snap,8475683.0,NaN,NaN,NaN,hit-crossbar,NaN,NaN,5 on 5,in net,104,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1


### SH%
- SH%: Percentage of Shots for that team that were Goals. GF*100/SF
- SV%: Percentage of Shots against that team that were not Goals. 100-(GA*100/SA)

In [106]:
sh_percent = all_shots_made[['game_id', 'season_id', 'team_id', 'typeDescKey']].copy()

# Create shot-type indicator columns
sh_percent['block'] = np.where(sh_percent['typeDescKey'] == 'blocked-shot', 1, 0)
sh_percent['goal'] = np.where(sh_percent['typeDescKey'] == 'goal', 1, 0)

# A "shot" is any attempt that is NOT blocked
# (blocked shots never reach the net, so they do not count toward SOG)
sh_percent['shots'] = np.where(sh_percent['block'] == 1, 0, 1)

# Group by season + team
sh_percent = (
    sh_percent
        .groupby(['season_id', 'team_id'], as_index=False)
        .agg(
            shots=('shots', 'sum'),
            goals=('goal', 'sum'),
        )
)

# Safe shooting percentage (avoids division-by-zero)
sh_percent['shot_pct'] = (
    sh_percent.apply(
        lambda row: round((row['goals'] * 100) / row['shots'], 2)
        if row['shots'] > 0 else 0,
        axis=1
    )
)


sh_percent.head()

,season_id,team_id,shots,goals,shot_pct
0,20252026,1.0,8070,650,8.05
1,20252026,2.0,8017,530,6.61
2,20252026,3.0,7970,444,5.57
3,20252026,4.0,6295,424,6.74
4,20252026,5.0,6968,622,8.93


In [107]:
shots_against = all_shots_made[['game_id','season_id', 'team_id', 'typeDescKey']].copy()

# Set up columns
shots_against['block'] = np.where(shots_against['typeDescKey'] == 'blocked-shot', 1, 0)
shots_against['goal'] = np.where(shots_against['typeDescKey'] == "goal", 1, 0)
shots_against['shot'] = np.where(shots_against['typeDescKey'] != "goal", 1, 0)
shots_against['shots'] = np.where(shots_against['block'] == 1, 0, 1)


# ----------------------------------------------------
# FIXED OPPONENT MAP (NO APPLY/STACK ERRORS)
# ----------------------------------------------------

# Extract the two team_ids per game
team_pairs = (
    shots_against[['game_id', 'team_id']]
    .drop_duplicates()
    .groupby('game_id')['team_id']
    .apply(list)
)

# Build opponent mapping as a dict-of-dicts
opponent_map = {}

for game_id, ids in team_pairs.items():
    if len(ids) == 2:
        opponent_map[game_id] = {
            ids[0]: ids[1],
            ids[1]: ids[0]
        }
    else:
        # fallback in case of weird data
        opponent_map[game_id] = {}


# Assign opponent team for each shot attempt
def get_opponent(row):
    lookup = opponent_map.get(row['game_id'], {})
    return lookup.get(row['team_id'])

shots_against['opponent_team_id'] = shots_against.apply(get_opponent, axis=1)

shots_against.head()

,game_id,season_id,team_id,typeDescKey,block,goal,shot,shots,opponent_team_id
0,2025020001,20252026,13.0,blocked-shot,1,0,1,0,16.0
1,2025020001,20252026,13.0,blocked-shot,1,0,1,0,16.0
2,2025020001,20252026,13.0,shot-on-goal,0,0,1,1,16.0
3,2025020001,20252026,13.0,shot-on-goal,0,0,1,1,16.0
4,2025020001,20252026,16.0,missed-shot,0,0,1,1,13.0


In [108]:

# ===============================
# Opponent / Against Shot Metrics
# ===============================

# Group by season + opponent team ID
against = (
    shots_against
        .groupby(['season_id', 'opponent_team_id'], as_index=False)
        .agg(
            shots_against=('shots', 'sum'),
            goals_against=('goal', 'sum')
        )
)

# Rename opponent_team_id → team_id (so it's consistent with your "for" dataset)
against = against.rename(columns={'opponent_team_id': 'team_id'})

# Safe shot percentage against
# (prevents division-by-zero warnings if a team allows 0 shots)
against['shot_pct_against'] = (
    against.apply(
        lambda row: round((row['goals_against'] * 100) / row['shots_against'], 2)
        if row['shots_against'] > 0 else 0,
        axis=1
    )
)

# Preview
against.head()



,season_id,team_id,shots_against,goals_against,shot_pct_against
0,20252026,1.0,7344,452,6.15
1,20252026,2.0,7448,487,6.54
2,20252026,3.0,7923,532,6.71
3,20252026,4.0,6378,488,7.65
4,20252026,5.0,6687,445,6.65


In [109]:


# Merge both DataFrames on team_id
shooting_stats = sh_percent.merge(against, on=['team_id','season_id'], how='left')

# Calculate Save Percentage (SV%)
shooting_stats['sv_pct'] = 100 - ((shooting_stats['goals_against'] * 100) / shooting_stats['shots_against'])

# Optional: round the SV% column
shooting_stats['sv_pct'] = shooting_stats['sv_pct'].round(2)

# Done!
shooting_stats.head()

,season_id,team_id,shots,goals,shot_pct,shots_against,goals_against,shot_pct_against,sv_pct
0,20252026,1.0,8070,650,8.05,7344,452,6.15,93.85
1,20252026,2.0,8017,530,6.61,7448,487,6.54,93.46
2,20252026,3.0,7970,444,5.57,7923,532,6.71,93.29
3,20252026,4.0,6295,424,6.74,6378,488,7.65,92.35
4,20252026,5.0,6968,622,8.93,6687,445,6.65,93.35


In [110]:
#PDO - Shooting percentage plus save percentage. (GF/SF)+(GA/SA)
shooting_stats['pdo'] = shooting_stats['shot_pct'] + shooting_stats['sv_pct']
shooting_stats = shooting_stats[['team_id', 'season_id', 'shot_pct', 'shots', 'shots_against', 'shot_pct_against', 'sv_pct', 'pdo']]
shooting_stats.head()

,team_id,season_id,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo
0,1.0,20252026,8.05,8070,7344,6.15,93.85,101.90
1,2.0,20252026,6.61,8017,7448,6.54,93.46,100.07
2,3.0,20252026,5.57,7970,7923,6.71,93.29,98.86
3,4.0,20252026,6.74,6295,6378,7.65,92.35,99.09
4,5.0,20252026,8.93,6968,6687,6.65,93.35,102.28


### Danger Zones

In [111]:
# Danger zone

# Define conditions based on the corrected MD and HD coordinates

# High Danger Zone (HD) - Right side of the rink
hd_condition_right = (
    (all_shots_made["x_coord"] >= 66.5)
    & (all_shots_made["x_coord"] <= 89)
    & (all_shots_made["y_coord"] >= -7)
    & (all_shots_made["y_coord"] <= 7)
)

# High Danger Zone (HD) - Left side of the rink (mirrored)
hd_condition_left = (
    (all_shots_made["x_coord"] >= -89)
    & (all_shots_made["x_coord"] <= -66.5)
    & (all_shots_made["y_coord"] >= -7)
    & (all_shots_made["y_coord"] <= 7)
)

# Medium Danger Zone (MD) - Right side
md_condition_right = (
    (
        (all_shots_made["x_coord"] >= 69)
        & (all_shots_made["x_coord"] <= 89)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= -7)
    )
    | (
        (all_shots_made["x_coord"] >= 54)
        & (all_shots_made["x_coord"] <= 69)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= 22)
    )
    | (
        (all_shots_made["x_coord"] >= 40)
        & (all_shots_made["x_coord"] <= 54)
        & (all_shots_made["y_coord"] >= -7)
        & (all_shots_made["y_coord"] <= 7)
    )
)

# Medium Danger Zone (MD) - Left side (mirrored)
md_condition_left = (
    (
        (all_shots_made["x_coord"] >= -89)
        & (all_shots_made["x_coord"] <= -69)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= -7)
    )
    | (
        (all_shots_made["x_coord"] >= -69)
        & (all_shots_made["x_coord"] <= -54)
        & (all_shots_made["y_coord"] >= -22)
        & (all_shots_made["y_coord"] <= 22)
    )
    | (
        (all_shots_made["x_coord"] >= -54)
        & (all_shots_made["x_coord"] <= -40)
        & (all_shots_made["y_coord"] >= -7)
        & (all_shots_made["y_coord"] <= 7)
    )
)

# Low Danger Zone (LD) - Every other coordinate
ld_condition = ~(
    hd_condition_right | hd_condition_left | md_condition_right | md_condition_left
)

# Apply conditions to classify shots
conditions = [
    hd_condition_right | hd_condition_left,
    md_condition_right | md_condition_left,
    ld_condition,
]
values = ["HD", "MD", "LD"]

all_shots_made["shot_danger"] = np.select(conditions, values, default="LD")

# Create new columns for HD, MD, and LD shots
all_shots_made["hd_shot"] = (all_shots_made["shot_danger"] == "HD").astype(int)
all_shots_made["md_shot"] = (all_shots_made["shot_danger"] == "MD").astype(int)
all_shots_made["ld_shot"] = (all_shots_made["shot_danger"] == "LD").astype(int)


In [112]:
all_shots_made["game_time"] = (
    all_shots_made["period_number"] - 1
) * 1200 + all_shots_made["game_in_seconds"]

# Sort by game_id and game_time
# all_shots_made = all_shots_made.sort_values(by=['game_id', 'game_time'])

# Calculate time difference from the last shot in the same game
all_shots_made["time_since_last_shot"] = all_shots_made.groupby("game_id")[
    "game_time"
].diff()

# Define a rebound as a shot that follows another shot within 3 seconds
all_shots_made["is_rebound"] = (all_shots_made["time_since_last_shot"] <= 3).astype(int)

all_shots_made.head()


,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound
0,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,MD,0,1,0,22,NaN,0
1,20252026,2025020001,67,01:13:00,18:47:00,508,blocked-shot,25,1,13.0,-64.0,-16.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8476891.0,NaN,5 on 5,in net,73,13,16,20252026,2025020001,70.0,N,giveaway,16.0,False,3.0,1.0,0,MD,0,1,0,73,51.0,0
2,20252026,2025020001,71,01:16:00,18:44:00,506,shot-on-goal,29,1,13.0,-58.0,-22.0,O,NaN,NaN,wrist,8481519.0,0.0,1.0,NaN,None,NaN,NaN,5 on 5,in net,76,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,MD,0,1,0,76,3.0,1
3,20252026,2025020001,102,01:35:00,18:25:00,506,shot-on-goal,35,1,13.0,-33.0,-19.0,O,NaN,NaN,slap,8481519.0,0.0,2.0,NaN,None,NaN,NaN,5 on 5,in net,95,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,LD,0,0,1,95,19.0,0
4,20252026,2025020001,80,01:44:00,18:16:00,507,missed-shot,38,1,16.0,55.0,-29.0,O,NaN,NaN,snap,8475683.0,NaN,NaN,NaN,hit-crossbar,NaN,NaN,5 on 5,in net,104,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,LD,0,0,1,104,9.0,0


### Scoring Chances

In [113]:

# Define conditions
conditions = [
    all_shots_made['hd_shot'] == 1,  # High Danger Shot
    all_shots_made['md_shot'] == 1,  # Medium Danger Shot
    all_shots_made['ld_shot'] == 1   # Low Danger Shot
]

# Define corresponding values
values = [3, 2, 1]  # Base values for scoring chances

# Apply np.select() to determine base scoring chance
all_shots_made['scoring_chance'] = np.select(conditions, values, default=1)

# Apply additional logic: If rush shot or rebound, increase scoring chance by 1
all_shots_made['scoring_chance'] += (all_shots_made['rush_shot'] == 1) | (all_shots_made['is_rebound'] == 1)
all_shots_made['scoring_chance'] -= all_shots_made['typeCode'] == 508

all_shots_made['scoring_chance'] = np.where(all_shots_made['scoring_chance'] >=2, 1, 0)
# Display result
all_shots_made.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound,scoring_chance
0,20252026,2025020001,101,00:22:00,19:38:00,508,blocked-shot,12,1,13.0,-61.0,3.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8482807.0,NaN,5 on 5,in net,22,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,MD,0,1,0,22,NaN,0,0
1,20252026,2025020001,67,01:13:00,18:47:00,508,blocked-shot,25,1,13.0,-64.0,-16.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8476891.0,NaN,5 on 5,in net,73,13,16,20252026,2025020001,70.0,N,giveaway,16.0,False,3.0,1.0,0,MD,0,1,0,73,51.0,0,1
2,20252026,2025020001,71,01:16:00,18:44:00,506,shot-on-goal,29,1,13.0,-58.0,-22.0,O,NaN,NaN,wrist,8481519.0,0.0,1.0,NaN,None,NaN,NaN,5 on 5,in net,76,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,MD,0,1,0,76,3.0,1,1
3,20252026,2025020001,102,01:35:00,18:25:00,506,shot-on-goal,35,1,13.0,-33.0,-19.0,O,NaN,NaN,slap,8481519.0,0.0,2.0,NaN,None,NaN,NaN,5 on 5,in net,95,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,LD,0,0,1,95,19.0,0,0
4,20252026,2025020001,80,01:44:00,18:16:00,507,missed-shot,38,1,16.0,55.0,-29.0,O,NaN,NaN,snap,8475683.0,NaN,NaN,NaN,hit-crossbar,NaN,NaN,5 on 5,in net,104,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,LD,0,0,1,104,9.0,0,0


In [114]:
scoring_chance_stats = all_shots_made[all_shots_made['scoring_chance']==1]
scoring_chance_stats.head()

,season_id,game_id,eventId,timeInPeriod,timeRemaining,typeCode,typeDescKey,sortOrder,period_number,team_id,x_coord,y_coord,zone_code,hitter,hittee,shot_type,goalie,away_sog,home_sog,player_id,reason,blocking_player,scoring_player,situation,goalie_situation,game_in_seconds,home_team,away_team,season_id_1,game_id_1,prev_event_time,prev_zone,prev_event_type,prev_team_id,prev_event_was_stoppage,time_diff,rush_shot,corsi,shot_danger,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound,scoring_chance
1,20252026,2025020001,67,01:13:00,18:47:00,508,blocked-shot,25,1,13.0,-64.0,-16.0,D,NaN,NaN,None,0.0,NaN,NaN,NaN,blocked,8476891.0,NaN,5 on 5,in net,73,13,16,20252026,2025020001,70.0,N,giveaway,16.0,False,3.0,1.0,0,MD,0,1,0,73,51.0,0,1
2,20252026,2025020001,71,01:16:00,18:44:00,506,shot-on-goal,29,1,13.0,-58.0,-22.0,O,NaN,NaN,wrist,8481519.0,0.0,1.0,NaN,None,NaN,NaN,5 on 5,in net,76,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,MD,0,1,0,76,3.0,1,1
9,20252026,2025020001,163,03:34:00,16:26:00,506,shot-on-goal,73,1,13.0,-58.0,28.0,O,NaN,NaN,slap,8481519.0,2.0,3.0,NaN,None,NaN,NaN,5 on 5,in net,214,13,16,20252026,2025020001,211.0,D,blocked-shot,13.0,False,3.0,1.0,1,LD,0,0,1,214,3.0,1,1
11,20252026,2025020001,172,03:55:00,16:05:00,508,blocked-shot,79,1,16.0,85.0,6.0,O,NaN,NaN,None,0.0,NaN,NaN,NaN,teammate-blocked,8483450.0,NaN,5 on 5,in net,235,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,HD,1,0,0,235,17.0,0,1
13,20252026,2025020001,181,04:41:00,15:19:00,507,missed-shot,88,1,13.0,-82.0,4.0,O,NaN,NaN,tip-in,8481519.0,NaN,NaN,NaN,wide-right,NaN,NaN,5 on 5,in net,281,13,16,20252026,2025020001,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,HD,1,0,0,281,35.0,0,1


In [115]:
scoring_chance_stats = scoring_chance_stats[['game_id', 'season_id','eventId', 'timeInPeriod', 'timeRemaining',  'team_id','typeCode', 'typeDescKey','shot_type',  'goalie',
        'situation', 'goalie_situation',
       'game_in_seconds', 'home_team', 'away_team', 'away_sog', 'home_sog','rush_shot', 'corsi',
       'hd_shot', 'md_shot', 'ld_shot', 'scoring_chance']]
scoring_chance_stats.head()

,game_id,season_id,eventId,timeInPeriod,timeRemaining,team_id,typeCode,typeDescKey,shot_type,goalie,situation,goalie_situation,game_in_seconds,home_team,away_team,away_sog,home_sog,rush_shot,corsi,hd_shot,md_shot,ld_shot,scoring_chance
1,2025020001,20252026,67,01:13:00,18:47:00,13.0,508,blocked-shot,None,0.0,5 on 5,in net,73,13,16,NaN,NaN,1.0,0,0,1,0,1
2,2025020001,20252026,71,01:16:00,18:44:00,13.0,506,shot-on-goal,wrist,8481519.0,5 on 5,in net,76,13,16,0.0,1.0,0.0,1,0,1,0,1
9,2025020001,20252026,163,03:34:00,16:26:00,13.0,506,shot-on-goal,slap,8481519.0,5 on 5,in net,214,13,16,2.0,3.0,1.0,1,0,0,1,1
11,2025020001,20252026,172,03:55:00,16:05:00,16.0,508,blocked-shot,None,0.0,5 on 5,in net,235,13,16,NaN,NaN,0.0,1,1,0,0,1
13,2025020001,20252026,181,04:41:00,15:19:00,13.0,507,missed-shot,tip-in,8481519.0,5 on 5,in net,281,13,16,NaN,NaN,0.0,1,1,0,0,1


In [116]:

# Default shooting team = team_id
scoring_chance_stats['shooting_team_id'] = scoring_chance_stats['team_id']

# Identify blocked shots (typeCode = 508)
mask = scoring_chance_stats['typeCode'] == 508

# For blocked shots, shooter is the OTHER team
home_block = (scoring_chance_stats['team_id'] == scoring_chance_stats['home_team'])
away_block = (scoring_chance_stats['team_id'] == scoring_chance_stats['away_team'])

# Shooter when home player blocked → shooter = away_team
scoring_chance_stats.loc[mask, 'shooting_team_id'] = scoring_chance_stats['away_team'].where(
    home_block, scoring_chance_stats['shooting_team_id']
)

# Shooter when away player blocked → shooter = home_team
scoring_chance_stats.loc[mask, 'shooting_team_id'] = scoring_chance_stats['home_team'].where(
    away_block, scoring_chance_stats['shooting_team_id']
)


scoring_chance_stats['scoring_chance_shot'] = np.where(
    scoring_chance_stats['typeCode'].isin([505, 506]),
    1, 
    0
)

scoring_chance_stats['scoring_chance_goal'] = np.where(
    scoring_chance_stats['typeCode'] == 505, 
    1, 
    0
)


scoring_chance_stats.head()

,game_id,season_id,eventId,timeInPeriod,timeRemaining,team_id,typeCode,typeDescKey,shot_type,goalie,situation,goalie_situation,game_in_seconds,home_team,away_team,away_sog,home_sog,rush_shot,corsi,hd_shot,md_shot,ld_shot,scoring_chance,shooting_team_id,scoring_chance_shot,scoring_chance_goal
1,2025020001,20252026,67,01:13:00,18:47:00,13.0,508,blocked-shot,None,0.0,5 on 5,in net,73,13,16,NaN,NaN,1.0,0,0,1,0,1,16.0,0,0
2,2025020001,20252026,71,01:16:00,18:44:00,13.0,506,shot-on-goal,wrist,8481519.0,5 on 5,in net,76,13,16,0.0,1.0,0.0,1,0,1,0,1,13.0,1,0
9,2025020001,20252026,163,03:34:00,16:26:00,13.0,506,shot-on-goal,slap,8481519.0,5 on 5,in net,214,13,16,2.0,3.0,1.0,1,0,0,1,1,13.0,1,0
11,2025020001,20252026,172,03:55:00,16:05:00,16.0,508,blocked-shot,None,0.0,5 on 5,in net,235,13,16,NaN,NaN,0.0,1,1,0,0,1,13.0,0,0
13,2025020001,20252026,181,04:41:00,15:19:00,13.0,507,missed-shot,tip-in,8481519.0,5 on 5,in net,281,13,16,NaN,NaN,0.0,1,1,0,0,1,13.0,0,0


In [117]:

# Make sure season_id is included
scf_stats = scoring_chance_stats[['season_id', 'shooting_team_id',
                                  'scoring_chance', 'scoring_chance_shot',
                                  'scoring_chance_goal']].copy()

# Group by season and team
scf_stats = (
    scf_stats
        .groupby(['season_id', 'shooting_team_id'], as_index=False)
        .agg(
            SCF=('scoring_chance', 'sum'),
            SCSF=('scoring_chance_shot', 'sum'),
            SCGF=('scoring_chance_goal', 'sum'),
        )
        .rename(columns={'shooting_team_id': 'team_id'})
)

# Preview
scf_stats.head()


,season_id,team_id,SCF,SCSF,SCGF
0,20252026,1.0,16269,3952,551
1,20252026,2.0,15613,3643,446
2,20252026,3.0,14988,3607,343
3,20252026,4.0,12816,3200,361
4,20252026,5.0,14150,3753,552


In [118]:

# --- Home SCA ---
sca_home = (
    scoring_chance_stats
        .groupby(['season_id', 'home_team'], as_index=False)
        .agg(SCA=('scoring_chance', 'sum'))
        .rename(columns={'home_team': 'team_id'})
)

# --- Away SCA ---
sca_away = (
    scoring_chance_stats
        .groupby(['season_id', 'away_team'], as_index=False)
        .agg(SCA=('scoring_chance', 'sum'))
        .rename(columns={'away_team': 'team_id'})
)

# --- Combine home + away ---
sca_stats = (
    pd.concat([sca_home, sca_away], ignore_index=True)
      .groupby(['season_id', 'team_id'], as_index=False)
      .agg(SCA=('SCA', 'sum'))
)

# --- Merge SCF + SCA ---
final_scoring_stats = (
    scf_stats.merge(sca_stats, on=['season_id', 'team_id'], how='left')
)

final_scoring_stats.head()

,season_id,team_id,SCF,SCSF,SCGF,SCA
0,20252026,1.0,16269,3952,551,31062
1,20252026,2.0,15613,3643,446,30342
2,20252026,3.0,14988,3607,343,30282
3,20252026,4.0,12816,3200,361,24645
4,20252026,5.0,14150,3753,552,26503


### Strength of Schedule

In [119]:

team_stats = []

def calculate_team_win_percentage(season_df, team_abbrev):
    # Wins (home or away)
    wins = (
        len(season_df[(season_df["awayTeam.abbrev"] == team_abbrev) &
                      (season_df["awayTeam.score"] > season_df["homeTeam.score"])])
        +
        len(season_df[(season_df["homeTeam.abbrev"] == team_abbrev) &
                      (season_df["homeTeam.score"] > season_df["awayTeam.score"])])
    )

    # Total games
    games = len(season_df[(season_df["awayTeam.abbrev"] == team_abbrev) |
                          (season_df["homeTeam.abbrev"] == team_abbrev)])

    return wins / games if games > 0 else 0


# Loop by season_id
for season_id, season_df in current_summary.groupby("season_id"):
    team_abbrevs = pd.concat([
        season_df["awayTeam.abbrev"],
        season_df["homeTeam.abbrev"]
    ]).unique()

    for team_abbrev in team_abbrevs:
        win_pct = calculate_team_win_percentage(season_df, team_abbrev)

        team_stats.append({
            "season_id": season_id,
            "tri_code": team_abbrev,
            "win_percentage": win_pct
        })

team_stats_df = pd.DataFrame(team_stats)

# ===============================================================
# STEP 2 — Map Win% back into the main games dataframe
# ===============================================================

current_summary = current_summary.merge(
    team_stats_df.rename(columns={"tri_code": "awayTeam.abbrev"}),
    on=["season_id", "awayTeam.abbrev"],
    how="left"
).rename(columns={"win_percentage": "awayTeam_win_percentage"})

current_summary = current_summary.merge(
    team_stats_df.rename(columns={"tri_code": "homeTeam.abbrev"}),
    on=["season_id", "homeTeam.abbrev"],
    how="left"
).rename(columns={"win_percentage": "homeTeam_win_percentage"})


# ===============================================================
# STEP 3 — Calculate SOS per season_id + team_id
# ===============================================================

# Away Team SOS = average win% of opponents (home teams)
away_team_sos = (
    current_summary.groupby(["season_id", "awayTeam.abbrev"])["homeTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"awayTeam.abbrev": "tri_code",
                     "homeTeam_win_percentage": "SOS"})
)

# Home Team SOS = average win% of opponents (away teams)
home_team_sos = (
    current_summary.groupby(["season_id", "homeTeam.abbrev"])["awayTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"homeTeam.abbrev": "tri_code",
                     "awayTeam_win_percentage": "SOS"})
)

# Combine home + away appearances
team_sos = pd.concat([away_team_sos, home_team_sos], ignore_index=True)

# Final SOS per season + team
team_sos = (
    team_sos.groupby(["season_id", "tri_code"], as_index=False)["SOS"]
    .mean()
)

# Final output
team_sos.head()

,season_id,tri_code,SOS
0,20252026,ANA,0.463436
1,20252026,BOS,0.482961
2,20252026,BUF,0.470956
3,20252026,CAR,0.454499
4,20252026,CBJ,0.448259


In [120]:
full_schedule = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\season_schedule.csv"

full_schedule_df = pd.read_csv(full_schedule, low_memory=False)
full_schedule_df = full_schedule_df[full_schedule_df['season']==20252026]
full_schedule_df = full_schedule_df.rename(columns={'season':'season_id'})


In [121]:

# Step 1: Filter out remaining games (future games only)
future_games = full_schedule_df[full_schedule_df['gameState']=="FUT"]
future_games = future_games.drop(columns=['index', 'awayTeam.logo', 'homeTeam.logo'])
future_games.head()

,game_id,season_id,gameType,awayTeam.id,awayTeam.abbrev,homeTeam.id,homeTeam.abbrev,homeTeam.placeName.default,awayTeam.placeName.default,awayTeam.score,homeTeam.score,winningGoalScorer.playerId,winningGoalie.playerId,gameState,link,game_date_time,start_time,game_date
349,2025020350,20252026,2,21,COL,16,CHI,Chicago,Colorado,NaN,NaN,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-11-23 19:00:00-05:00,7:00 pm,2025-11-23
350,2025020351,20252026,2,6,BOS,28,SJS,San Jose,Boston,NaN,NaN,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-11-23 20:00:00-05:00,8:00 pm,2025-11-23
351,2025020352,20252026,2,20,CGY,23,VAN,Vancouver,Calgary,NaN,NaN,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-11-23 21:00:00-05:00,9:00 pm,2025-11-23
352,2025020353,20252026,2,4,PHI,14,TBL,Tampa Bay,Philadelphia,NaN,NaN,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-11-24 19:00:00-05:00,7:00 pm,2025-11-24
353,2025020354,20252026,2,17,DET,1,NJD,New Jersey,Detroit,NaN,NaN,NaN,NaN,FUT,https://api-web.nhle.com/v1/gamecenter/2025020...,2025-11-24 19:00:00-05:00,7:00 pm,2025-11-24


In [122]:

# Build lookup keyed by (season_id, team_abbrev aka tri_code)
lookup = team_stats_df.set_index(["season_id", "tri_code"])["win_percentage"]

# Build key pairs for mapping:
away_keys = list(zip(future_games["season_id"], future_games["awayTeam.abbrev"]))
home_keys = list(zip(future_games["season_id"], future_games["homeTeam.abbrev"]))

# Apply mapping:
future_games["awayTeam_win_percentage"] = [lookup.get(k, np.nan) for k in away_keys]
future_games["homeTeam_win_percentage"] = [lookup.get(k, np.nan) for k in home_keys]

# ======================================================
# Step 3 — Compute Remaining SOS for home + away teams
# ======================================================

# Away team SOS (they are playing the home's win%)
away_team_remaining_sos = (
    future_games.groupby(["season_id", "awayTeam.abbrev"])["homeTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"awayTeam.abbrev": "tri_code",
                     "homeTeam_win_percentage": "Remaining_SOS"})
)

# Home team SOS (they are playing the away's win%)
home_team_remaining_sos = (
    future_games.groupby(["season_id", "homeTeam.abbrev"])["awayTeam_win_percentage"]
    .mean()
    .reset_index()
    .rename(columns={"homeTeam.abbrev": "tri_code",
                     "awayTeam_win_percentage": "Remaining_SOS"})
)


# ======================================================
# Step 4 — Combine home + away and final aggregation
# ======================================================

remaining_team_sos = pd.concat(
    [away_team_remaining_sos, home_team_remaining_sos],
    axis=0,
    ignore_index=True
)

# Final average remaining SOS per team per season
remaining_team_sos = (
    remaining_team_sos
    .groupby(["season_id", "tri_code"], as_index=False)["Remaining_SOS"]
    .mean()
    .sort_values(["season_id", "tri_code"])
)

remaining_team_sos.head()


,season_id,tri_code,Remaining_SOS
0,20252026,ANA,0.428987
1,20252026,BOS,0.448022
2,20252026,BUF,0.463759
3,20252026,CAR,0.452051
4,20252026,CBJ,0.471011


In [123]:
team_sos.head()

,season_id,tri_code,SOS
0,20252026,ANA,0.463436
1,20252026,BOS,0.482961
2,20252026,BUF,0.470956
3,20252026,CAR,0.454499
4,20252026,CBJ,0.448259


In [124]:

strength = team_sos.merge(remaining_team_sos, on=['season_id','tri_code'], how = 'inner')
strength = strength.merge(teams, on='tri_code', how='left')
strength.head()



,season_id,tri_code,SOS,Remaining_SOS,team_id
0,20252026,ANA,0.463436,0.428987,24
1,20252026,BOS,0.482961,0.448022,6
2,20252026,BUF,0.470956,0.463759,7
3,20252026,CAR,0.454499,0.452051,12
4,20252026,CBJ,0.448259,0.471011,29


### Special Teams


In [125]:
current_pk = all_pk[all_pk['season_id']==20252026]
current_pp = all_pp[all_pp['season_id']==20252026]

In [126]:
pp_num_cols = ['team_id', 'season_id',
    'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
    'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame',
    'ppOpportunities', 'ppOpportunitiesPerGame',
    'ppTimeOnIcePerGame', 'shGoalsAgainst',
    'shGoalsAgainstPerGame'
]

pk_num_cols = ['team_id', 'season_id',
    'penaltyKillNetPct', 'penaltyKillPct',
       'pkNetGoals', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points',
       'pointsPct', 'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame', 'teamFullName', 'ties',
       'timesShorthanded', 'timesShorthandedPerGame', 'wins']

current_pp[pp_num_cols] = current_pp[pp_num_cols].fillna(0)
current_pk[pk_num_cols] = current_pk[pk_num_cols].fillna(0)

C:\Users\kjchr\AppData\Local\Temp\ipykernel_7884\1808792292.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  current_pk[pk_num_cols] = current_pk[pk_num_cols].fillna(0)


In [127]:
pp_stats = current_pp.copy()

pp_stats = pp_stats[['game_id', 'season_id', 'team_id', 'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
       'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame', 'ppOpportunities',
       'ppOpportunitiesPerGame', 'ppTimeOnIcePerGame', 'shGoalsAgainst',
       'shGoalsAgainstPerGame', 'opponentTeamAbbrev']]

pp_stats.head()

,game_id,season_id,team_id,powerPlayGoalsFor,powerPlayNetPct,powerPlayPct,ppGoalsPerGame,ppNetGoals,ppNetGoalsPerGame,ppOpportunities,ppOpportunitiesPerGame,ppTimeOnIcePerGame,shGoalsAgainst,shGoalsAgainstPerGame,opponentTeamAbbrev
29292,2025020001,20252026,16,0,0.000000,0.00000,0.0,0,0.0,3,3.0,360.0,0,0.0,FLA
29293,2025020001,20252026,13,1,0.500000,0.50000,1.0,1,1.0,2,2.0,222.0,0,0.0,CHI
29294,2025020002,20252026,5,0,0.000000,0.00000,0.0,0,0.0,1,1.0,120.0,0,0.0,NYR
29295,2025020002,20252026,3,0,0.000000,0.00000,0.0,0,0.0,2,2.0,240.0,0,0.0,PIT
29296,2025020003,20252026,21,1,0.166666,0.16666,1.0,1,1.0,6,6.0,605.0,0,0.0,LAK


In [128]:
# ==============================================================
# AGGREGATE PP STATS BY TEAM + SEASON
# ==============================================================

pp_stats = pp_stats[['game_id', 'season_id', 'team_id', 
                     'powerPlayGoalsFor', 'powerPlayNetPct', 'powerPlayPct',
                     'ppGoalsPerGame', 'ppNetGoals', 'ppNetGoalsPerGame',
                     'ppOpportunities', 'ppOpportunitiesPerGame',
                     'ppTimeOnIcePerGame', 'shGoalsAgainst',
                     'shGoalsAgainstPerGame', 'opponentTeamAbbrev']].copy()

# --------------------------------------------------------------
# 1. Aggregate raw totals (correct way)
# --------------------------------------------------------------
agg_pp = (
    pp_stats.groupby(['season_id', 'team_id'], as_index=False)
    .agg(
        games_played=('game_id', 'nunique'),
        powerPlayGoalsFor=('powerPlayGoalsFor', 'sum'),
        shGoalsAgainst=('shGoalsAgainst', 'sum'),
        ppOpportunities=('ppOpportunities', 'sum'),
        ppNetGoals=('ppNetGoals', 'sum'),
        ppTimeOnIce=('ppTimeOnIcePerGame', 'sum'),
    )
)

# --------------------------------------------------------------
# 2. Recompute correct percentages 
# --------------------------------------------------------------

# PP% = PPGF / PPO
agg_pp['powerPlayPct'] = (
    agg_pp['powerPlayGoalsFor'] / agg_pp['ppOpportunities']
).replace([np.inf, -np.inf], np.nan).round(4) * 100

# Net PP% = (PPGF - SHGA) / PPO
agg_pp['powerPlayNetPct'] = (
    (agg_pp['powerPlayGoalsFor'] - agg_pp['shGoalsAgainst'])
    / agg_pp['ppOpportunities']
).replace([np.inf, -np.inf], np.nan).round(4) * 100

# --------------------------------------------------------------
# 3. Recompute per-game metrics
# --------------------------------------------------------------

agg_pp['ppGoalsPerGame'] = (
    agg_pp['powerPlayGoalsFor'] / agg_pp['games_played']
).round(3)

agg_pp['ppOpportunitiesPerGame'] = (
    agg_pp['ppOpportunities'] / agg_pp['games_played']
).round(3)

agg_pp['shGoalsAgainstPerGame'] = (
    agg_pp['shGoalsAgainst'] / agg_pp['games_played']
).round(3)

agg_pp['ppNetGoalsPerGame'] = (
    agg_pp['ppNetGoals'] / agg_pp['games_played']
).round(3)

agg_pp['ppTimeOnIcePerGame'] = (
    agg_pp['ppTimeOnIce'] / agg_pp['games_played']
).round(3)

# Clean final DataFrame
numeric_cols = agg_pp.select_dtypes(include=['float']).columns
agg_pp[numeric_cols] = agg_pp[numeric_cols].round(3)

# Final dataset
agg_pp.head()


,season_id,team_id,games_played,powerPlayGoalsFor,shGoalsAgainst,ppOpportunities,ppNetGoals,ppTimeOnIce,powerPlayPct,powerPlayNetPct,ppGoalsPerGame,ppOpportunitiesPerGame,shGoalsAgainstPerGame,ppNetGoalsPerGame,ppTimeOnIcePerGame
0,20252026,1,23,14,0,58,14,5824.0,24.14,24.14,0.609,2.522,0.000,0.609,253.217
1,20252026,2,24,10,3,79,7,7843.0,12.66,8.86,0.417,3.292,0.125,0.292,326.792
2,20252026,3,25,11,2,56,9,5831.0,19.64,16.07,0.440,2.240,0.080,0.360,233.240
3,20252026,4,22,10,2,59,8,5719.0,16.95,13.56,0.455,2.682,0.091,0.364,259.955
4,20252026,5,22,16,2,51,14,4984.0,31.37,27.45,0.727,2.318,0.091,0.636,226.545


In [129]:
pk_stats = current_pk.copy()


pk_stats = pk_stats[['game_id', 'season_id', 'team_id',  'penaltyKillNetPct', 'penaltyKillPct',
       'pkNetGoals', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points',
       'pointsPct', 'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame','timesShorthanded', 'timesShorthandedPerGame', 'opponentTeamAbbrev']]

pk_stats.head()

,game_id,season_id,team_id,penaltyKillNetPct,penaltyKillPct,pkNetGoals,pkNetGoalsPerGame,pkTimeOnIcePerGame,points,pointsPct,ppGoalsAgainst,ppGoalsAgainstPerGame,shGoalsFor,shGoalsForPerGame,timesShorthanded,timesShorthandedPerGame,opponentTeamAbbrev
29292,2025020001,20252026,13,1.00,1.00,0,0.0,360.0,2,1.0,0,0.0,0,0.0,3,3.0,CHI
29293,2025020001,20252026,16,0.50,0.50,-1,-1.0,222.0,0,0.0,1,1.0,0,0.0,2,2.0,FLA
29294,2025020002,20252026,3,1.00,1.00,0,0.0,120.0,0,0.0,0,0.0,0,0.0,1,1.0,PIT
29295,2025020002,20252026,5,1.00,1.00,0,0.0,240.0,2,1.0,0,0.0,0,0.0,2,2.0,NYR
29296,2025020003,20252026,21,0.75,0.75,-1,-1.0,438.0,2,1.0,1,1.0,0,0.0,4,4.0,LAK


In [130]:
# ==============================================
# AGGREGATE PK STATS BY TEAM + SEASON
# ==============================================

# Start with your selected PK stats
pk_stats = pk_stats[['game_id', 'season_id', 'team_id', 'penaltyKillNetPct', 'penaltyKillPct',
       'pkNetGoals', 'pkNetGoalsPerGame', 'pkTimeOnIcePerGame', 'points',
       'pointsPct', 'ppGoalsAgainst', 'ppGoalsAgainstPerGame', 'shGoalsFor',
       'shGoalsForPerGame','timesShorthanded', 'timesShorthandedPerGame', 'opponentTeamAbbrev']].copy()

# Group by team + season
agg_pk = (
    pk_stats.groupby(['season_id', 'team_id'], as_index=False)
    .agg(
        games_played=('game_id', 'nunique'),
        ppGoalsAgainst=('ppGoalsAgainst', 'sum'),
        shGoalsFor=('shGoalsFor', 'sum'),
        timesShorthanded=('timesShorthanded', 'sum'),
        pkNetGoals=('pkNetGoals', 'sum'),
        pkTimeOnIce=('pkTimeOnIcePerGame', 'sum'),   # total minutes short-handed
        points=('points', 'sum'),
    )
)

# ----------------------------------------------
# RE-CALCULATE ALL % METRICS PROPERLY
# ----------------------------------------------

# Classic PK% = 1 - PPGA / TSH
agg_pk['penaltyKillPct'] = (
    1 - (agg_pk['ppGoalsAgainst'] / agg_pk['timesShorthanded'])
).round(4) * 100

# Net PK% = (TSH - PPGA + SHGF) / TSH
agg_pk['penaltyKillNetPct'] = (
    (agg_pk['timesShorthanded'] - agg_pk['ppGoalsAgainst'] + agg_pk['shGoalsFor'])
    / agg_pk['timesShorthanded']
).round(4) * 100

# Per-game metrics (recomputed)
agg_pk['ppGoalsAgainstPerGame'] = agg_pk['ppGoalsAgainst'] / agg_pk['games_played']
agg_pk['shGoalsForPerGame'] = agg_pk['shGoalsFor'] / agg_pk['games_played']
agg_pk['timesShorthandedPerGame'] = agg_pk['timesShorthanded'] / agg_pk['games_played']
agg_pk['pkNetGoalsPerGame'] = agg_pk['pkNetGoals'] / agg_pk['games_played']
agg_pk['pkTimeOnIcePerGame'] = agg_pk['pkTimeOnIce'] / agg_pk['games_played']

# Points%
agg_pk['pointsPct'] = agg_pk['points'] / (agg_pk['games_played'] * 2)

# Round final numbers
numeric_cols = agg_pk.select_dtypes(include=['float']).columns
agg_pk[numeric_cols] = agg_pk[numeric_cols].round(3)

# Final dataset
agg_pk.head()


,season_id,team_id,games_played,ppGoalsAgainst,shGoalsFor,timesShorthanded,pkNetGoals,pkTimeOnIce,points,penaltyKillPct,penaltyKillNetPct,ppGoalsAgainstPerGame,shGoalsForPerGame,timesShorthandedPerGame,pkNetGoalsPerGame,pkTimeOnIcePerGame,pointsPct
0,20252026,1,23,12,4,68,-8,6971.0,31,82.35,88.24,0.522,0.174,2.957,-0.348,303.087,0.674
1,20252026,2,24,12,5,72,-7,6994.0,28,83.33,90.28,0.500,0.208,3.000,-0.292,291.417,0.583
2,20252026,3,25,15,2,74,-13,7533.0,26,79.73,82.43,0.600,0.080,2.960,-0.520,301.320,0.520
3,20252026,4,22,9,0,65,-9,6731.0,27,86.15,86.15,0.409,0.000,2.955,-0.409,305.955,0.614
4,20252026,5,22,9,1,61,-8,6353.0,27,85.25,86.89,0.409,0.045,2.773,-0.364,288.773,0.614


In [131]:
special_teams = agg_pp.merge(
    agg_pk,
    on=['season_id', 'team_id'],
    how='inner',
    suffixes=('_pp', '_pk')   # optional — only needed if any column names overlap
)

special_teams.head()

,season_id,team_id,games_played_pp,powerPlayGoalsFor,shGoalsAgainst,ppOpportunities,ppNetGoals,ppTimeOnIce,powerPlayPct,powerPlayNetPct,ppGoalsPerGame,ppOpportunitiesPerGame,shGoalsAgainstPerGame,ppNetGoalsPerGame,ppTimeOnIcePerGame,games_played_pk,ppGoalsAgainst,shGoalsFor,timesShorthanded,pkNetGoals,pkTimeOnIce,points,penaltyKillPct,penaltyKillNetPct,ppGoalsAgainstPerGame,shGoalsForPerGame,timesShorthandedPerGame,pkNetGoalsPerGame,pkTimeOnIcePerGame,pointsPct
0,20252026,1,23,14,0,58,14,5824.0,24.14,24.14,0.609,2.522,0.000,0.609,253.217,23,12,4,68,-8,6971.0,31,82.35,88.24,0.522,0.174,2.957,-0.348,303.087,0.674
1,20252026,2,24,10,3,79,7,7843.0,12.66,8.86,0.417,3.292,0.125,0.292,326.792,24,12,5,72,-7,6994.0,28,83.33,90.28,0.500,0.208,3.000,-0.292,291.417,0.583
2,20252026,3,25,11,2,56,9,5831.0,19.64,16.07,0.440,2.240,0.080,0.360,233.240,25,15,2,74,-13,7533.0,26,79.73,82.43,0.600,0.080,2.960,-0.520,301.320,0.520
3,20252026,4,22,10,2,59,8,5719.0,16.95,13.56,0.455,2.682,0.091,0.364,259.955,22,9,0,65,-9,6731.0,27,86.15,86.15,0.409,0.000,2.955,-0.409,305.955,0.614
4,20252026,5,22,16,2,51,14,4984.0,31.37,27.45,0.727,2.318,0.091,0.636,226.545,22,9,1,61,-8,6353.0,27,85.25,86.89,0.409,0.045,2.773,-0.364,288.773,0.614


In [132]:
special_teams = special_teams.rename(
            columns={"teamId": "team_id",
                "powerPlayGoalsFor": "pp_gf",
                "powerPlayNetPct": "pp_net_pct",
                "powerPlayPct": "pp_pct",
                "ppGoalsPerGame": "ppg_g",
                "ppNetGoals": "ppg_net",
                "ppNetGoalsPerGame": "ppg_net_g",
                "ppOpportunities": "pp_opp",
                "ppOpportunitiesPerGame": "pp_opp_g",
                "penaltyKillNetPct": "pk_net_pct",
                "penaltyKillPct": "pk_pct",
                "pkNetGoals": "pkg_net",
                "pkNetGoalsPerGame": "pkg_net_g",
                "ppGoalsAgainst": "ppga",
                "ppGoalsAgainstPerGame": "ppga_g",
                "shGoalsFor": "shgf",
                "shGoalsForPerGame": "shgf_g",
            }
        )

special_teams = special_teams.drop(columns=['points'])
special_teams.head()

,season_id,team_id,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct
0,20252026,1,23,14,0,58,14,5824.0,24.14,24.14,0.609,2.522,0.000,0.609,253.217,23,12,4,68,-8,6971.0,82.35,88.24,0.522,0.174,2.957,-0.348,303.087,0.674
1,20252026,2,24,10,3,79,7,7843.0,12.66,8.86,0.417,3.292,0.125,0.292,326.792,24,12,5,72,-7,6994.0,83.33,90.28,0.500,0.208,3.000,-0.292,291.417,0.583
2,20252026,3,25,11,2,56,9,5831.0,19.64,16.07,0.440,2.240,0.080,0.360,233.240,25,15,2,74,-13,7533.0,79.73,82.43,0.600,0.080,2.960,-0.520,301.320,0.520
3,20252026,4,22,10,2,59,8,5719.0,16.95,13.56,0.455,2.682,0.091,0.364,259.955,22,9,0,65,-9,6731.0,86.15,86.15,0.409,0.000,2.955,-0.409,305.955,0.614
4,20252026,5,22,16,2,51,14,4984.0,31.37,27.45,0.727,2.318,0.091,0.636,226.545,22,9,1,61,-8,6353.0,85.25,86.89,0.409,0.045,2.773,-0.364,288.773,0.614


## Final DF

In [133]:

current_tx_standings = current_tx_standings.merge(teams, on='tri_code', how='left')

In [134]:
final_df = (
    current_tx_standings[
        ['season_id', 'tri_code', 'team_id', 'team_name', 'conferenceName',
       'conferenceSequence', 'gamesPlayed', 'points', 'winPctg', 'pointPctg',
       'regulationPlusOtWins', 'goalDifferential', 'goalFor', 'goalAgainst',
       'homeGamesPlayed', 'homeWins', 'roadGamesPlayed', 'roadWins',
       'l10GoalDifferential', 'streakCode', 'streakCount', 'home_win_pctg',
       'road_win_pctg', 'gf_g', 'ga_g', 'projected_points',
       'thanksgiving_date']
    ]
    .merge(strength, on=['team_id','tri_code','season_id'], how='left')
    .merge(special_teams, on=['team_id','season_id'], how="left")
    .merge(rush_stats, on=['team_id','tri_code','season_id'], how='left')

    .merge(team_goalie_stats, on=['team_id','tri_code','season_id'], how='left')
    .merge(shooting_stats,on=['team_id','season_id'], how='left')
    .merge(final_scoring_stats, on=['team_id','season_id'], how='left')
)
today = date.today()
final_df['run_date'] = today
final_df.head()

,season_id,tri_code,team_id,team_name,conferenceName,conferenceSequence,gamesPlayed,points,winPctg,pointPctg,regulationPlusOtWins,goalDifferential,goalFor,goalAgainst,homeGamesPlayed,homeWins,roadGamesPlayed,roadWins,l10GoalDifferential,streakCode,streakCount,home_win_pctg,road_win_pctg,gf_g,ga_g,projected_points,thanksgiving_date,SOS,Remaining_SOS,games_played_pp,pp_gf,shGoalsAgainst,pp_opp,ppg_net,ppTimeOnIce,pp_pct,pp_net_pct,ppg_g,pp_opp_g,shGoalsAgainstPerGame,ppg_net_g,ppTimeOnIcePerGame,games_played_pk,ppga,shgf,timesShorthanded,pkg_net,pkTimeOnIce,pk_pct,pk_net_pct,ppga_g,shgf_g,timesShorthandedPerGame,pkg_net_g,pkTimeOnIcePerGame,pointsPct,rush_shots_for,rush_shots_against,rush_shot_differential,team_savePct,team_GAA,shot_pct,shots,shots_against,shot_pct_against,sv_pct,pdo,SCF,SCSF,SCGF,SCA,run_date
0,20252026,COL,21,Colorado Avalanche,Western,1,23,39,0.739130,0.847826,17,44,94,50,11,9,12,8,32,W,10,0.818182,0.666667,4.086957,2.173913,139.0,2025-11-27,0.458574,0.434816,23,14,2,87,12,8996.0,16.09,13.79,0.609,3.783,0.087,0.522,391.130,23,9,1,73,-8,7623.0,87.67,89.04,0.391,0.043,3.174,-0.348,331.435,0.848,2630,2590,40,0.919594,2.069236,8.64,8745,6970,4.81,95.19,103.83,14361,4587,669,30370,2025-12-04
1,20252026,DAL,25,Dallas Stars,Western,2,24,34,0.625000,0.708333,13,16,83,67,11,6,13,9,20,W,2,0.545455,0.692308,3.458333,2.791667,116.0,2025-11-27,0.438614,0.447594,24,25,4,80,21,7285.0,31.25,26.25,1.042,3.333,0.167,0.875,303.542,24,17,0,77,-17,7561.0,77.92,77.92,0.708,0.000,3.208,-0.708,315.042,0.708,1917,1939,-22,0.906483,2.556874,7.30,6757,7528,7.19,92.81,100.11,14898,3500,397,27745,2025-12-04
2,20252026,NJD,1,New Jersey Devils,Eastern,1,23,31,0.652174,0.673913,13,3,72,69,10,9,13,6,-3,W,2,0.900000,0.461538,3.130435,3.000000,111.0,2025-11-27,0.471750,0.442211,23,14,0,58,14,5824.0,24.14,24.14,0.609,2.522,0.000,0.609,253.217,23,12,4,68,-8,6971.0,82.35,88.24,0.522,0.174,2.957,-0.348,303.087,0.674,2498,2500,-2,0.895472,2.888793,8.05,8070,7344,6.15,93.85,101.90,16269,3952,551,31062,2025-12-04
3,20252026,TBL,14,Tampa Bay Lightning,Eastern,2,23,30,0.608696,0.652174,14,13,74,61,13,8,10,6,12,W,5,0.615385,0.600000,3.217391,2.652174,107.0,2025-11-27,0.463503,0.447731,23,12,2,74,10,7648.0,16.22,13.51,0.522,3.217,0.087,0.435,332.522,23,9,2,77,-7,7615.0,88.31,90.91,0.391,0.087,3.348,-0.304,331.087,0.652,2088,2104,-16,0.910233,2.534346,6.42,7184,7460,5.74,94.26,100.68,13442,3061,385,26776,2025-12-04
4,20252026,CAR,12,Carolina Hurricanes,Eastern,3,23,30,0.608696,0.652174,13,11,80,69,10,6,13,8,-2,L,2,0.600000,0.615385,3.478261,3.000000,107.0,2025-11-27,0.454499,0.452051,23,10,2,68,8,6869.0,14.71,11.76,0.435,2.957,0.087,0.348,298.652,23,15,3,71,-12,6983.0,78.87,83.10,0.652,0.130,3.087,-0.522,303.609,0.652,2964,2913,51,0.890999,2.724320,6.29,8822,7103,7.73,92.27,98.56,13049,3722,433,30254,2025-12-04


In [135]:

# Define export path
export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Thanksgiving Prep\tx_curr_df.csv"

# Ensure the folder exists before writing the file
os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # Export the data to CSV
final_df.to_csv(export_path, index=False)
